# Timan-Pechora / Timana models

This notebook was previously exported from Google Colab (paths like `/content/...`).

Changes in this revision:
- Use local repo imports (`src/rockphysx`) instead of loading loose `.py` files from `/content`.
- Replace `/content/...` paths with repo-relative paths.
- Clear saved outputs for a smaller, cleaner notebook in git.

**Note:** if you still get errors mentioning `/content`, you are running an old Colab snippet or stale notebook state. Re-open this notebook, restart the kernel, and run cells top-to-bottom (or `Run All`).


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


def find_upwards(filename: str, start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for p in (start, *start.parents):
        candidate = p / filename
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not find {filename!r} starting from {start}")


REPO_ROOT = find_upwards("pyproject.toml").parent
SRC_DIR = REPO_ROOT / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import numpy as np

from rockphysx.models.emt.dem_transport_elastic import (
    dem_elastic_velocities,
    velocities_from_moduli,
)
from rockphysx.models.emt.dem_thermal import dem_thermal_conductivity
from rockphysx.models.emt.sca_thermal import sca_effective_conductivity
from rockphysx.models.emt.gsa_transport import two_phase_thermal_isotropic
from rockphysx.models.emt.bruggeman import bruggeman_isotropic


def depolarization_factors_spheroid(aspect_ratio: float | np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return (n1, n3) depolarization factors for a spheroid (a1=a2, a3)."""
    r = np.asarray(aspect_ratio, dtype=float)
    if np.any(~np.isfinite(r)) or np.any(r <= 0):
        raise ValueError("Aspect ratio must be positive and finite.")

    n3 = np.empty_like(r)
    sphere = np.isclose(r, 1.0)
    oblate = r < 1.0
    prolate = r > 1.0

    n3[sphere] = 1.0 / 3.0

    rr = r[oblate]
    if rr.size > 0:
        xi = np.sqrt(np.maximum(1.0 / (rr * rr) - 1.0, 0.0))
        n3[oblate] = ((1.0 + xi * xi) / (xi**3)) * (xi - np.arctan(xi))

    rr = r[prolate]
    if rr.size > 0:
        e = np.sqrt(np.maximum(1.0 - 1.0 / (rr * rr), 0.0))
        n3[prolate] = ((1.0 - e * e) / (2.0 * e**3)) * (np.log((1.0 + e) / (1.0 - e)) - 2.0 * e)

    n1 = (1.0 - n3) / 2.0
    return n1, n3


def mt_transport_single_aspect_ratio(
    phi: float,
    aspect_ratio: float | np.ndarray,
    prop_matrix: float,
    prop_fluid: float,
) -> float | np.ndarray:
    """Mori–Tanaka transport (scalar property) for randomly oriented spheroidal inclusions."""
    phi = float(phi)
    if not (0.0 <= phi < 1.0):
        raise ValueError(f"Porosity must be in [0,1). Got {phi}")

    r = np.asarray(aspect_ratio, dtype=float)
    n1, n3 = depolarization_factors_spheroid(r)

    dm = float(prop_matrix)
    df = float(prop_fluid)
    delta = df - dm

    a1 = dm / (dm + n1 * delta)
    a3 = dm / (dm + n3 * delta)
    a_bar = (2.0 * a1 + a3) / 3.0

    c_i = phi
    c_m = 1.0 - phi
    value = dm + c_i * delta * a_bar / (c_m + c_i * a_bar)

    if np.ndim(value) == 0:
        return float(value)
    return value

import os
import tempfile


def ensure_writable_dir(preferred: Path, *, name: str = "results") -> Path:
    """Create and return a writable directory.

    Tries (in order): env override -> preferred -> CWD/name -> tmp/name.
    This avoids Colab-style `/content/...` paths failing on read-only filesystems.
    """

    env = os.environ.get("THESIS_RP_OUTDIR") or os.environ.get("OUTDIR")
    candidates: list[Path] = []
    if env:
        candidates.append(Path(env).expanduser())
    candidates.extend([
        Path(preferred),
        Path.cwd() / name,
        Path(tempfile.gettempdir()) / name,
    ])

    errors: list[str] = []
    for candidate in candidates:
        try:
            candidate.mkdir(parents=True, exist_ok=True)
            test_file = candidate / ".write_test"
            test_file.write_text("ok")
            test_file.unlink(missing_ok=True)
            return candidate
        except Exception as e:
            errors.append(f"{candidate}: {type(e).__name__}: {e}")

    raise OSError(
        "Could not create a writable output directory. Tried:\n" + "\n".join(errors)
    )


RESULTS_ROOT = ensure_writable_dir(REPO_ROOT / "results", name="results")


### Carbonates

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd

# =========================
# CONFIG
# =========================

DEFAULT_MATRIX_TC = 2.98
DEFAULT_AIR_TC = 0.025
DEFAULT_OIL_TC = 0.13
DEFAULT_BRINE_AVG_TC = 0.60

DEFAULT_SHEET = "All properties_data"

# Укажи путь к своему Excel после загрузки в Colab
EXCEL_PATH = str(REPO_ROOT / "Tver_ver1.xlsx")
SHEET_NAME = DEFAULT_SHEET

# Названия колонок в Excel
ID_COL = "Sample"
PHI_COL = "Porosity,%"

# Если пористость в процентах, поставить True
POROSITY_IN_PERCENT = True

# Состояния: имя состояния, колонка с экспериментальным TC, теплопроводность флюида
STATE_ORDER = [
    ("dry", "TC air", DEFAULT_AIR_TC),
    ("oil", "TC oil", DEFAULT_OIL_TC),
    ("brine_avg", "TC 60", DEFAULT_BRINE_AVG_TC),
]

OUTDIR = RESULTS_ROOT / "results_comparison"
OUTDIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def clip_phi(phi: float) -> float:
    return float(np.clip(phi, 0.0, 1.0))


def normalize_porosity(series: pd.Series, in_percent: bool = True) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    if in_percent:
        s = s / 100.0
    return s


# -------------------------
# Wiener bounds
# -------------------------
def wiener_parallel(lambda_m: float, lambda_f: float, phi: float) -> float:
    phi = clip_phi(phi)
    return (1.0 - phi) * lambda_m + phi * lambda_f


def wiener_series(lambda_m: float, lambda_f: float, phi: float) -> float:
    phi = clip_phi(phi)
    denom = (1.0 - phi) / lambda_m + phi / lambda_f
    return 1.0 / denom


# -------------------------
# Lichtenecker = geometric mean
# -------------------------
def lichtenecker(lambda_m: float, lambda_f: float, phi: float) -> float:
    phi = clip_phi(phi)
    return (lambda_m ** (1.0 - phi)) * (lambda_f ** phi)


# -------------------------
# Roy-Adler
# -------------------------
def roy_adler(lambda_m: float, lambda_f: float, phi: float) -> float:
    phi = clip_phi(phi)
    return ((1.0 - phi) * np.sqrt(lambda_m) + phi * np.sqrt(lambda_f)) ** 2


# -------------------------
# Hashin-Shtrikman bounds
# Generic implementation for two-phase isotropic conductivity
# -------------------------
def hs_bounds(lambda_m: float, lambda_f: float, phi: float) -> tuple[float, float]:
    phi = clip_phi(phi)

    if np.isclose(lambda_m, lambda_f):
        return lambda_m, lambda_m

    # Sort phases by conductivity
    k1 = max(lambda_m, lambda_f)   # higher conductivity
    k2 = min(lambda_m, lambda_f)   # lower conductivity

    # Volume fraction of high-conductivity phase
    if lambda_m >= lambda_f:
        c1 = 1.0 - phi   # matrix fraction
        c2 = phi         # pore fluid fraction
    else:
        c1 = phi
        c2 = 1.0 - phi

    hs_upper = k1 + c2 / (1.0 / (k2 - k1) + c1 / (3.0 * k1))
    hs_lower = k2 + c1 / (1.0 / (k1 - k2) + c2 / (3.0 * k2))
    return hs_lower, hs_upper


def hs_upper(lambda_m: float, lambda_f: float, phi: float) -> float:
    return hs_bounds(lambda_m, lambda_f, phi)[1]


def hs_lower(lambda_m: float, lambda_f: float, phi: float) -> float:
    return hs_bounds(lambda_m, lambda_f, phi)[0]


# -------------------------
# Krupiczka
# epsilon = porosity
# log10 version
# -------------------------
def krupiczka(lambda_s: float, lambda_f: float, epsilon: float) -> float:
    epsilon = clip_phi(epsilon)
    epsilon = max(epsilon, 1e-8)
    ratio = lambda_s / lambda_f
    exponent = 0.28 - 0.757 * np.log10(epsilon) - 0.057 * np.log10(ratio)
    return lambda_f * (ratio ** exponent)


# -------------------------
# Zimmerman (1989)
# Generic formula with selectable pore-shape beta
# Here I add spherical pores as the default comparison variant.
# -------------------------
def zimmerman_beta(r: float, pore_shape: str = "spherical") -> float:
    """
    r = lambda_f / lambda_s

    pore_shape:
        - "thin_cracks"
        - "spherical"
        - "needle_like"
    """
    r = float(r)

    if r <= 0:
        return np.nan

    if pore_shape == "thin_cracks":
        # alpha -> 0
        return ((1.0 - r) * (1.0 + 2.0 * r)) / (3.0 * r)

    elif pore_shape == "spherical":
        # alpha = 1
        return 3.0 * (1.0 - r) / (2.0 + r)

    elif pore_shape == "needle_like":
        # alpha -> infinity
        return ((1.0 - r) * (5.0 + r)) / (3.0 * (1.0 + r))

    else:
        raise ValueError(f"Unknown pore_shape: {pore_shape}")


def zimmerman(lambda_s: float, lambda_f: float, phi: float, pore_shape: str = "spherical") -> float:
    """
    Zimmerman (1989):
        lambda = lambda_s * [ ((1-phi)(1-r) + r*beta*phi) /
                              ((1-phi)(1-r) + beta*phi) ]
    where r = lambda_f / lambda_s
    """
    phi = clip_phi(phi)

    if lambda_s <= 0 or lambda_f <= 0:
        return np.nan

    if np.isclose(lambda_s, lambda_f):
        return lambda_s

    r = lambda_f / lambda_s
    beta_val = zimmerman_beta(r, pore_shape=pore_shape)

    numerator = (1.0 - phi) * (1.0 - r) + r * beta_val * phi
    denominator = (1.0 - phi) * (1.0 - r) + beta_val * phi

    if np.isclose(denominator, 0.0):
        return np.nan

    return lambda_s * (numerator / denominator)


def zimmerman_spherical(lambda_s: float, lambda_f: float, phi: float) -> float:
    return zimmerman(lambda_s, lambda_f, phi, pore_shape="spherical")


# Optional alternatives if later you want to compare pore-shape end-members:
def zimmerman_thin_cracks(lambda_s: float, lambda_f: float, phi: float) -> float:
    return zimmerman(lambda_s, lambda_f, phi, pore_shape="thin_cracks")


def zimmerman_needle_like(lambda_s: float, lambda_f: float, phi: float) -> float:
    return zimmerman(lambda_s, lambda_f, phi, pore_shape="needle_like")


# -------------------------
# Albert et al. (2017)
# Original model was calibrated for air- and water-saturated rocks.
# Here:
#   - for dry/air we use Eq. (19) with S = 1.3 * phi^0.62
#   - for fluid-saturated states we use Eq. (18)
# Using it for oil is an extrapolation beyond the original calibration.
# -------------------------
def albert_saturation_parameter(phi: float) -> float:
    phi = clip_phi(phi)
    return float(np.clip(1.3 * (phi ** 0.62), 0.0, 1.0))


def albert_saturated(lambda_s: float, lambda_f: float, phi: float, epsilon: float = 1.0) -> float:
    phi = clip_phi(phi)
    return (
        lambda_s * (((1.0 - phi) / (1.0 + phi)) ** 1.08)
        + lambda_f * (phi ** 0.4)
    ) * epsilon


def albert_dry(lambda_s: float, lambda_f: float, phi: float, epsilon: float = 1.0) -> float:
    phi = clip_phi(phi)
    S = albert_saturation_parameter(phi)
    return albert_saturated(lambda_s, lambda_f, phi, epsilon=epsilon) * (1.0 - S)


def albert_2017(lambda_s: float, lambda_f: float, phi: float) -> float:
    """
    Practical wrapper for your current comparison workflow:
    - if fluid is air -> use dry equation
    - otherwise -> use saturated equation
    """
    phi = clip_phi(phi)

    # dry / air case
    if np.isclose(lambda_f, DEFAULT_AIR_TC, rtol=0.0, atol=1e-12):
        return albert_dry(lambda_s, lambda_f, phi, epsilon=1.0)

    # saturated case (water in original paper; oil here is extrapolation)
    return albert_saturated(lambda_s, lambda_f, phi, epsilon=1.0)


# -------------------------
# Modified Maxwell / Eucken (1940)
# First modified Maxwell model
# -------------------------
def modified_maxwell(lambda_s: float, lambda_f: float, phi: float) -> float:
    phi = clip_phi(phi)

    if lambda_s <= 0 or lambda_f <= 0:
        return np.nan

    ratio = lambda_s / lambda_f
    eps = (1.0 - ratio) / (2.0 * ratio + 1.0)

    denom = 1.0 - phi * eps
    if np.isclose(denom, 0.0):
        return np.nan

    return lambda_s * ((1.0 + 2.0 * phi * eps) / denom)


# -------------------------
# Ribaud (1937)
# -------------------------
def ribaud(lambda_s: float, lambda_f: float, phi: float) -> float:
    phi = clip_phi(phi)
    return lambda_f * (phi ** (2.0 / 3.0)) + lambda_s * (1.0 - phi ** (2.0 / 3.0))


# -------------------------
# Mendel (1997)
# -------------------------
def mendel(lambda_s: float, lambda_f: float, phi: float) -> float:
    phi = clip_phi(phi)

    if lambda_s <= 0 or lambda_f <= 0:
        return np.nan

    denom = 2.0 * lambda_s + lambda_f
    if np.isclose(denom, 0.0):
        return np.nan

    return lambda_s * (1.0 - (3.0 * phi * (lambda_s - lambda_f) / denom))


# -------------------------
# Sidorov (1979)
# -------------------------
def sidorov(lambda_s: float, lambda_f: float, phi: float) -> float:
    phi = clip_phi(phi)

    if lambda_s <= 0 or lambda_f <= 0:
        return np.nan

    denom = 1.0 + ((lambda_s / lambda_f) - 1.0) * phi
    if np.isclose(denom, 0.0):
        return np.nan

    return lambda_s * (1.0 / denom)

In [ ]:
PREDICTOR_MODELS = {
    "Lichtenecker (geometric mean)": lichtenecker,
    "Roy-Adler": roy_adler,
    "Krupiczka": krupiczka,
    "Zimmerman (1989; spherical pores)": zimmerman_spherical,
    "Zimmerman (1989; thin cracks)": zimmerman_thin_cracks,
    "Zimmerman (1989; needle-like pores)": zimmerman_needle_like,
    # "Albert et al. (2017; empirical air/water)": albert_2017,
    "Modified Maxwell (Eucken, 1940)": modified_maxwell,
    "Ribaud (1937)": ribaud,
    "Mendel (1997)": mendel,
    "Sidorov (1979)": sidorov,
}

In [ ]:
MODEL_ALLOWED_STATES = {
    "Lichtenecker (geometric mean)": {"dry", "oil", "brine_avg"},
    "Roy-Adler": {"dry", "oil", "brine_avg"},
    "Krupiczka": {"dry", "oil", "brine_avg"},
    "Zimmerman (1989; spherical pores)": {"dry", "oil", "brine_avg"},
    "Zimmerman (1989; thin cracks)": {"dry", "oil", "brine_avg"},
    "Zimmerman (1989; needle-like pores)": {"dry", "oil", "brine_avg"},
    # "Albert et al. (2017; empirical air/water)": {"dry", "brine_avg"},
    "Modified Maxwell (Eucken, 1940)": {"dry", "oil", "brine_avg"},
    "Ribaud (1937)": {"dry", "brine_avg"},
    "Mendel (1997)": {"dry", "brine_avg"},
    "Sidorov (1979)": {"dry", "brine_avg"},
}

In [ ]:
df = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME)

df = df.copy()
df[PHI_COL] = normalize_porosity(df[PHI_COL], in_percent=POROSITY_IN_PERCENT)

display(df.head())
print("Rows:", len(df))
print("Columns:", list(df.columns))

In [ ]:
def evaluate_models(
    df: pd.DataFrame,
    id_col: str,
    phi_col: str,
    state_order,
    matrix_tc: float,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    predictor_rows = []
    bounds_rows = []

    for _, row in df.iterrows():
        sample_id = row[id_col]
        phi = row[phi_col]

        if pd.isna(phi):
            continue

        phi = float(phi)

        for state_name, meas_col, fluid_tc in state_order:
            if meas_col not in df.columns:
                continue
            if pd.isna(row[meas_col]):
                continue

            observed = float(row[meas_col])

            # -------------------------
            # predictor models
            # -------------------------
            for model_name, model_func in PREDICTOR_MODELS.items():
                if state_name not in MODEL_ALLOWED_STATES.get(model_name, set()):
                    continue

                predicted = model_func(matrix_tc, fluid_tc, phi)
                error = predicted - observed
                abs_error = abs(error)
                ape = abs_error / abs(observed) * 100.0 if observed != 0 else np.nan

                predictor_rows.append({
                    "sample_id": sample_id,
                    "state": state_name,
                    "measurement_col": meas_col,
                    "phi": phi,
                    "matrix_tc": matrix_tc,
                    "fluid_tc": fluid_tc,
                    "model": model_name,
                    "observed_tc": observed,
                    "predicted_tc": predicted,
                    "error": error,
                    "abs_error": abs_error,
                    "abs_pct_error": ape,
                })

            # -------------------------
            # bounds
            # -------------------------
            bound_values = {
                "wiener_parallel": wiener_parallel(matrix_tc, fluid_tc, phi),
                "wiener_series": wiener_series(matrix_tc, fluid_tc, phi),
                "hs_upper": hs_upper(matrix_tc, fluid_tc, phi),
                "hs_lower": hs_lower(matrix_tc, fluid_tc, phi),
            }

            bounds_rows.append({
                "sample_id": sample_id,
                "state": state_name,
                "measurement_col": meas_col,
                "phi": phi,
                "matrix_tc": matrix_tc,
                "fluid_tc": fluid_tc,
                "observed_tc": observed,
                **bound_values,
                "obs_inside_wiener": (
                    min(bound_values["wiener_series"], bound_values["wiener_parallel"])
                    <= observed <=
                    max(bound_values["wiener_series"], bound_values["wiener_parallel"])
                ),
                "obs_inside_hs": (
                    min(bound_values["hs_lower"], bound_values["hs_upper"])
                    <= observed <=
                    max(bound_values["hs_lower"], bound_values["hs_upper"])
                ),
            })

    predictor_df = pd.DataFrame(predictor_rows)
    bounds_df = pd.DataFrame(bounds_rows)

    summary_df = (
        predictor_df
        .groupby(["state", "model"], dropna=False)
        .agg(
            n=("observed_tc", "size"),
            mean_obs=("observed_tc", "mean"),
            mean_pred=("predicted_tc", "mean"),
            bias=("error", "mean"),
            mae=("abs_error", "mean"),
            rmse=("error", lambda s: np.sqrt(np.mean(np.square(s)))),
            mape=("abs_pct_error", "mean"),
        )
        .reset_index()
        .sort_values(["state", "rmse", "mae"])
    )

    return predictor_df, bounds_df, summary_df


predictor_df, bounds_df, summary_df = evaluate_models(
    df=df,
    id_col=ID_COL,
    phi_col=PHI_COL,
    state_order=STATE_ORDER,
    matrix_tc=DEFAULT_MATRIX_TC,
)

In [ ]:
display(summary_df)
display(bounds_df.head())
# display(predictor_df.head())

In [ ]:
from pathlib import Path
import tempfile

# Make saving robust even if cells were run out of order.
# Prefer RESULTS_ROOT/REPO_ROOT if available; otherwise fall back to CWD/tmp.
_results_root = None
if "RESULTS_ROOT" in globals():
    _results_root = Path(RESULTS_ROOT)
elif "REPO_ROOT" in globals():
    _results_root = Path(REPO_ROOT) / "results"
else:
    _results_root = Path.cwd() / "results"

safe_outdir = _results_root / "results_comparison"
try:
    safe_outdir.mkdir(parents=True, exist_ok=True)
    test_file = safe_outdir / ".write_test"
    test_file.write_text("ok")
    test_file.unlink(missing_ok=True)
except Exception:
    safe_outdir = Path(tempfile.gettempdir()) / "results_comparison"
    safe_outdir.mkdir(parents=True, exist_ok=True)

OUTDIR = safe_outdir

predictor_path = OUTDIR / "engineering_predictors_detailed.csv"
bounds_path = OUTDIR / "engineering_bounds_detailed.csv"
summary_path = OUTDIR / "engineering_predictors_summary.csv"

predictor_df.to_csv(predictor_path, index=False)
bounds_df.to_csv(bounds_path, index=False)
summary_df.to_csv(summary_path, index=False)

print("Saved:")
print(predictor_path)
print(bounds_path)
print(summary_path)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

states = ["dry", "oil", "brine_avg"]

for state in states:
    state_df = predictor_df[predictor_df["state"] == state].copy()
    if state_df.empty:
        print(f"No data for state: {state}")
        continue

    plt.figure(figsize=(7, 7))

    models = sorted(state_df["model"].dropna().unique())
    for model in models:
        sub = state_df[state_df["model"] == model]
        plt.scatter(
            sub["observed_tc"],
            sub["predicted_tc"],
            label=model,
            alpha=0.8,
        )

    all_obs = state_df["observed_tc"].to_numpy()
    all_pred = state_df["predicted_tc"].to_numpy()

    vmin = min(np.nanmin(all_obs), np.nanmin(all_pred))
    vmax = max(np.nanmax(all_obs), np.nanmax(all_pred))

    plt.plot([vmin, vmax], [vmin, vmax], linestyle="--")
    plt.xlabel("Observed thermal conductivity")
    plt.ylabel("Predicted thermal conductivity")
    plt.title(f"Predicted vs Observed — {state}")
    plt.legend()
    plt.xlim(0, 3.5)
    plt.ylim(0, 3.5)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
plt.figure(figsize=(8, 8))

models = sorted(predictor_df["model"].dropna().unique())
for model in models:
    sub = predictor_df[predictor_df["model"] == model]
    plt.scatter(
        sub["observed_tc"],
        sub["predicted_tc"],
        label=model,
        alpha=0.7,
    )

all_obs = predictor_df["observed_tc"].to_numpy()
all_pred = predictor_df["predicted_tc"].to_numpy()

vmin = min(np.nanmin(all_obs), np.nanmin(all_pred))
vmax = max(np.nanmax(all_obs), np.nanmax(all_pred))

plt.plot([vmin, vmax], [vmin, vmax], linestyle="--")
plt.xlabel("Observed thermal conductivity")
plt.ylabel("Predicted thermal conductivity")
plt.title("Predicted vs Observed — all states")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# Настройки
# -----------------------------
STATES_TO_PLOT = ["dry", "oil", "brine_avg"]
POROSITY_AXIS_IN_PERCENT = True
USE_DATA_PHI_RANGE = True
YMIN, YMAX = 0.0, 3.5

STATE_MAP = {state: (meas_col, fluid_tc) for state, meas_col, fluid_tc in STATE_ORDER}

CURVE_MODELS = {
    "Lichtenecker (geometric mean)": (lichtenecker, "red"),
    "Roy-Adler": (roy_adler, "green"),
    "Zimmerman (1989; spherical pores)": (zimmerman_spherical, "purple"),
    "Albert et al. (2017; empirical air/water)": (albert_2017, "blue"),
    "Modified Maxwell (Eucken, 1940)": (modified_maxwell, "orange"),
    "Ribaud (1937)": (ribaud, "brown"),
    "Mendel (1997)": (mendel, "magenta"),
    "Sidorov (1979)": (sidorov, "teal"),
}
BOUND_CURVES = {
    "wiener_parallel": wiener_parallel,
    "wiener_series": wiener_series,
    "hs_upper": hs_upper,
    "hs_lower": hs_lower,
}


def get_phi_range(df, phi_col, n=300):
    phi = pd.to_numeric(df[phi_col], errors="coerce").dropna().to_numpy()
    phi = phi[(phi >= 0) & (phi <= 1)]

    if len(phi) == 0:
        return np.linspace(0.0, 0.4, n)

    phi_min = 0.0
    phi_max = float(np.nanmax(phi))

    if np.isclose(phi_max, 0.0):
        phi_max = 0.4

    return np.linspace(phi_min, phi_max, n)


phi_grid = get_phi_range(df, PHI_COL, n=300)

fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)

for ax, state in zip(axes, STATES_TO_PLOT):
    if state not in STATE_MAP:
        ax.set_visible(False)
        continue

    meas_col, fluid_tc = STATE_MAP[state]

    if meas_col not in df.columns:
        ax.set_visible(False)
        continue

    state_df = df[[ID_COL, PHI_COL, meas_col]].copy().dropna(subset=[PHI_COL, meas_col])

    if state_df.empty:
        ax.set_visible(False)
        continue

    if POROSITY_AXIS_IN_PERCENT:
        x_obs = state_df[PHI_COL].to_numpy() * 100.0
        x_grid = phi_grid * 100.0
        xlabel = "Porosity, %"
    else:
        x_obs = state_df[PHI_COL].to_numpy()
        x_grid = phi_grid
        xlabel = "Porosity, fraction"

    # Эксперимент
    ax.scatter(
        x_obs,
        state_df[meas_col].to_numpy(),
        label="experiment",
        alpha=0.9,
        marker="o",
    )

    # Основные модели
    for model_name, (model_func, color) in CURVE_MODELS.items():
        if state not in MODEL_ALLOWED_STATES.get(model_name, set()):
            continue

        y = np.array([model_func(DEFAULT_MATRIX_TC, fluid_tc, phi) for phi in phi_grid])
        ax.plot(
            x_grid,
            y,
            linewidth=2.2,
            color=color,
            label=model_name,
            zorder=4,
        )

    # Wiener bounds
    for model_name in ["wiener_parallel", "wiener_series"]:
        y = [BOUND_CURVES[model_name](DEFAULT_MATRIX_TC, fluid_tc, phi) for phi in phi_grid]
        ax.plot(x_grid, y, linestyle="--", label=model_name)

    # HS bounds
    for model_name in ["hs_upper", "hs_lower"]:
        y = [BOUND_CURVES[model_name](DEFAULT_MATRIX_TC, fluid_tc, phi) for phi in phi_grid]
        ax.plot(x_grid, y, linestyle="-.", label=model_name)

    ax.set_title(state)
    ax.set_xlabel(xlabel)
    ax.set_ylim(YMIN, YMAX)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Thermal conductivity, W/(m·K)")

# Общая легенда
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.08))

plt.tight_layout()
plt.show()

In [ ]:
# fig.savefig("engineering_models_tc_vs_porosity_25.png", dpi=600, bbox_inches="tight")
# fig.savefig("engineering_models_tc_vs_porosity_25.pdf", bbox_inches="tight")

In [ ]:
def make_summary_by_state(
    summary_df: pd.DataFrame,
    state_order_out: list[str] | None = None,
    state_rename: dict[str, str] | None = None,
) -> pd.DataFrame:
    metrics_order = ["n", "mean_pred", "bias", "mae", "rmse", "mape"]

    out = summary_df.copy()

    # Если порядок явно не задан — берем все состояния из summary_df
    if state_order_out is None:
        state_order_out = list(out["state"].dropna().unique())

    # Переименования состояний
    if state_rename is None:
        state_rename = {
            "dry": "dry",
            "brine_avg": "brine",
            "oil": "oil",
        }

    # Оставляем все нужные состояния
    out = out[out["state"].isin(state_order_out)].copy()

    # Переименовываем только те, что есть в словаре; остальные оставляем как есть
    out["state"] = out["state"].map(lambda x: state_rename.get(x, x))

    table = (
        out.pivot_table(
            index="model",
            columns="state",
            values=metrics_order,
            aggfunc="first",
        )
    )

    # сначала состояние, потом метрика
    table = table.swaplevel(0, 1, axis=1)

    # порядок столбцов
    renamed_state_order = [state_rename.get(s, s) for s in state_order_out]

    desired_cols = []
    for state in renamed_state_order:
        for metric in metrics_order:
            desired_cols.append((state, metric))

    existing_cols = [col for col in desired_cols if col in table.columns]
    table = table.reindex(columns=pd.MultiIndex.from_tuples(existing_cols))

    # округление
    for col in table.columns:
        if col[1] == "n":
            table[col] = table[col].astype("Int64")
        else:
            table[col] = table[col].round(3)

    return table

In [ ]:
summary_by_state = make_summary_by_state(
    summary_df,
    state_order_out=["dry", "brine_avg", "oil"],
    state_rename={
        "dry": "dry",
        "brine_avg": "brine",
        "oil": "oil",
    },
)
display(summary_by_state)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd

import matplotlib.font_manager as fm

fonts = sorted({f.name for f in fm.fontManager.ttflist})
[name for name in fonts if "Times" in name]

# -----------------------------
# Шрифт
# -----------------------------
mpl.rcParams["font.family"] = "serif"
mpl.rcParams["font.serif"] = ["Times New Roman", "Times", "DejaVu Serif"]

# -----------------------------
# Настройки
# -----------------------------
STATES_TO_PLOT = ["dry", "oil", "brine_avg"]
POROSITY_AXIS_IN_PERCENT = True
YMIN, YMAX = 0.0, 3.5
XMIN, XMAX = 0.0, 25.0

STATE_MAP = {state: (meas_col, fluid_tc) for state, meas_col, fluid_tc in STATE_ORDER}

STATE_TITLES = {
    "dry": "Dry",
    "oil": "Oil-saturated",
    "brine_avg": "Brine-saturated",
}

CURVE_MODELS = {
    "Lichtenecker (geometric mean)": (lichtenecker, "red"),
    "Roy-Adler": (roy_adler, "green"),
    "Zimmerman (1989; spherical pores)": (zimmerman_spherical, "purple"),
    "Albert et al. (2017; empirical air/water)": (albert_2017, "blue"),
    "Modified Maxwell (Eucken, 1940)": (modified_maxwell, "orange"),
    "Ribaud (1937)": (ribaud, "brown"),
    "Mendel (1997)": (mendel, "magenta"),
    "Sidorov (1979)": (sidorov, "teal"),
}

def get_phi_range(phi_max_percent=25.0, n=400):
    phi_max = phi_max_percent / 100.0
    return np.linspace(0.0, phi_max, n)

phi_grid = get_phi_range(phi_max_percent=25.0, n=400)

# -----------------------------
# Стиль
# -----------------------------
plt.rcParams.update({
    "font.size": 13,
    "axes.titlesize": 15,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "mathtext.fontset": "stix",
})

fig, axes = plt.subplots(1, 3, figsize=(18, 5.8), sharey=True)

for ax, state in zip(axes, STATES_TO_PLOT):
    if state not in STATE_MAP:
        ax.set_visible(False)
        continue

    meas_col, fluid_tc = STATE_MAP[state]

    if meas_col not in df.columns:
        ax.set_visible(False)
        continue

    state_df = df[[ID_COL, PHI_COL, meas_col]].copy().dropna(subset=[PHI_COL, meas_col])
    state_df = state_df[state_df[PHI_COL] <= 0.25].copy()

    if state_df.empty:
        ax.set_visible(False)
        continue

    if POROSITY_AXIS_IN_PERCENT:
        x_obs = state_df[PHI_COL].to_numpy() * 100.0
        x_grid = phi_grid * 100.0
        xlabel = "Porosity, %"
    else:
        x_obs = state_df[PHI_COL].to_numpy()
        x_grid = phi_grid
        xlabel = "Porosity, fraction"

    wiener_upper_vals = np.array([wiener_parallel(DEFAULT_MATRIX_TC, fluid_tc, phi) for phi in phi_grid])
    wiener_lower_vals = np.array([wiener_series(DEFAULT_MATRIX_TC, fluid_tc, phi) for phi in phi_grid])

    hs_upper_vals = np.array([hs_upper(DEFAULT_MATRIX_TC, fluid_tc, phi) for phi in phi_grid])
    hs_lower_vals = np.array([hs_lower(DEFAULT_MATRIX_TC, fluid_tc, phi) for phi in phi_grid])

    ax.fill_between(
        x_grid,
        np.minimum(wiener_lower_vals, wiener_upper_vals),
        np.maximum(wiener_lower_vals, wiener_upper_vals),
        alpha=0.18,
        label="Wiener bounds",
    )

    ax.fill_between(
        x_grid,
        np.minimum(hs_lower_vals, hs_upper_vals),
        np.maximum(hs_lower_vals, hs_upper_vals),
        alpha=0.30,
        label="Hashin–Shtrikman bounds",
    )

    ax.scatter(
        x_obs,
        state_df[meas_col].to_numpy(),
        color="black",
        alpha=0.5,
        s=36,
        label="Experimental data",
        zorder=5,
    )

    for model_name, (model_func, color) in CURVE_MODELS.items():
        if state not in MODEL_ALLOWED_STATES.get(model_name, set()):
            continue

        y = np.array([model_func(DEFAULT_MATRIX_TC, fluid_tc, phi) for phi in phi_grid])
        ax.plot(
            x_grid,
            y,
            linewidth=2.2,
            color=color,
            label=model_name,
            zorder=4,
        )

    ax.set_title(STATE_TITLES.get(state, state))
    ax.set_xlabel(xlabel)
    ax.set_ylim(YMIN, YMAX)
    ax.set_xlim(XMIN, XMAX)
    ax.grid(True, alpha=0.25)

axes[0].set_ylabel("Thermal conductivity, W/(m·K)")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, -0.08),
)



fig.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()

fig.savefig("engineering_models_tc_vs_porosity_25.png", dpi=600, bbox_inches="tight")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# -----------------------------
# Настройки
# -----------------------------
STATES_TO_PLOT = ["dry", "oil", "brine_avg"]
POROSITY_AXIS_IN_PERCENT = True
ABS_REL_ERROR = False   # True -> |pred-obs|/obs * 100%
YMIN, YMAX = -20, 20    # можно поменять под ваши данные

STATE_MAP = {state: (meas_col, fluid_tc) for state, meas_col, fluid_tc in STATE_ORDER}

ERROR_MODELS = {
    "lichtenecker": lichtenecker,
    "roy_adler": roy_adler,
}

fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)

for ax, state in zip(axes, STATES_TO_PLOT):
    if state not in STATE_MAP:
        ax.set_visible(False)
        continue

    meas_col, fluid_tc = STATE_MAP[state]

    if meas_col not in df.columns:
        ax.set_visible(False)
        continue

    state_df = df[[ID_COL, PHI_COL, meas_col]].copy().dropna(subset=[PHI_COL, meas_col])

    if state_df.empty:
        ax.set_visible(False)
        continue

    phi_obs = state_df[PHI_COL].to_numpy(dtype=float)
    obs_tc = state_df[meas_col].to_numpy(dtype=float)

    if POROSITY_AXIS_IN_PERCENT:
        x = phi_obs * 100.0
        xlabel = "Porosity, %"
    else:
        x = phi_obs
        xlabel = "Porosity, fraction"

    for model_name, model_func in ERROR_MODELS.items():
        pred_tc = np.array([
            model_func(DEFAULT_MATRIX_TC, fluid_tc, phi) for phi in phi_obs
        ], dtype=float)

        rel_err = (pred_tc - obs_tc) / obs_tc * 100.0
        if ABS_REL_ERROR:
            rel_err = np.abs(rel_err)

        ax.scatter(
            x,
            rel_err,
            label=model_name,
            alpha=0.85,
        )

    ax.axhline(0.0, linestyle="--")
    ax.set_title(state)
    ax.set_xlabel(xlabel)
    ax.set_ylim(YMIN, YMAX)
    ax.set_xlim(left=0.0)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Relative prediction error, %")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.05))

plt.tight_layout()
plt.show()

### EMT

In [ ]:
!pip -q install numpy pandas scipy openpyxl

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar

DEFAULT_MATRIX_TC = 2.98
DEFAULT_AIR_TC = 0.025
DEFAULT_OIL_TC = 0.13
DEFAULT_BRINE_AVG_TC = 0.60
DEFAULT_ALPHA_BOUNDS = (1e-4, 1.0)
CALIBRATION_OBJECTIVE_MODE = "log_sse"  # sse | rel_sse | log_sse
ALPHA_FIT_MODE = "per_sample"  # per_sample | global


ID_COL = "Sample"
PHI_COL = "Porosity,%"
POROSITY_IN_PERCENT = True

STATE_ORDER = [
    ("dry", "TC air", DEFAULT_AIR_TC),
    ("oil", "TC oil", DEFAULT_OIL_TC),
    ("brine_avg", "TC 60", DEFAULT_BRINE_AVG_TC),
]

CALIBRATION_STATE = "dry"
# Joint inversion: set to ["dry","oil","brine_avg"] to fit one alpha per sample using all 3 states
CALIBRATION_STATES = [CALIBRATION_STATE]

# Limit EMT evaluation to these models (recommended for thesis tables)
EMT_MODELS_TO_EVAL = ["GSA", "SCA", "Bruggeman"]

OUTDIR = RESULTS_ROOT / "results_emt"
OUTDIR.mkdir(parents=True, exist_ok=True)

In [ ]:
EXCEL_PATH = str(REPO_ROOT / "Tver_ver1.xlsx")
SHEET_NAME = "All properties_data"

df = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME).copy()

if POROSITY_IN_PERCENT:
    df[PHI_COL] = pd.to_numeric(df[PHI_COL], errors="coerce") / 100.0
else:
    df[PHI_COL] = pd.to_numeric(df[PHI_COL], errors="coerce")

display(df.head())
print("Rows:", len(df))

In [ ]:
def mt_predict(matrix_tc: float, fluid_tc: float, phi: float, alpha: float) -> float:
    return float(
        mt_transport_single_aspect_ratio(
            phi=phi,
            aspect_ratio=alpha,
            prop_matrix=matrix_tc,
            prop_fluid=fluid_tc,
        )
    )

def sca_predict(matrix_tc: float, fluid_tc: float, phi: float, alpha: float) -> float:
    return float(
        sca_effective_conductivity(
            matrix_conductivity=matrix_tc,
            inclusion_conductivity=fluid_tc,
            inclusion_fraction=phi,
            aspect_ratio=alpha,
        )
    )

def dem_predict(matrix_tc: float, fluid_tc: float, phi: float, alpha: float) -> float:
    return float(
        dem_thermal_conductivity(
            matrix_conductivity=matrix_tc,
            inclusion_conductivity=fluid_tc,
            inclusion_fraction=phi,
            aspect_ratio=alpha,
        )
    )

def gsa_predict(matrix_tc: float, fluid_tc: float, phi: float, alpha: float) -> float:
    return float(
        two_phase_thermal_isotropic(
            matrix_value=matrix_tc,
            inclusion_value=fluid_tc,
            porosity=phi,
            aspect_ratio=alpha,
            comparison="matrix",   # можно потом заменить на "self_consistent" или "bayuk_linear_mix"
        )
    )

def bruggeman_predict(matrix_tc: float, fluid_tc: float, phi: float, alpha: float | None = None) -> float:
    return float(
        bruggeman_isotropic(
            volume_fractions=[1.0 - phi, phi],
            values=[matrix_tc, fluid_tc],
        )
    )


EMT_MODEL_SPECS = {
    "MT": {"func": mt_predict, "uses_alpha": True},
    "SCA": {"func": sca_predict, "uses_alpha": True},
    "DEM": {"func": dem_predict, "uses_alpha": True},
    "GSA": {"func": gsa_predict, "uses_alpha": True},
    "Bruggeman": {"func": bruggeman_predict, "uses_alpha": False},
}

In [ ]:
def quick_sanity_check(model_specs, matrix_tc=DEFAULT_MATRIX_TC, fluid_tc=DEFAULT_AIR_TC, alpha=0.1):
    rows = []
    for model_name, spec in model_specs.items():
        func = spec["func"]
        uses_alpha = spec["uses_alpha"]
        try:
            if uses_alpha:
                value = func(matrix_tc, fluid_tc, 0.0, alpha)
            else:
                value = func(matrix_tc, fluid_tc, 0.0, None)

            rows.append({
                "model": model_name,
                "phi0_value": value,
                "expected_matrix_tc": matrix_tc,
                "phi0_ok": np.isclose(value, matrix_tc, rtol=1e-6, atol=1e-6),
            })
        except Exception as e:
            rows.append({
                "model": model_name,
                "phi0_value": np.nan,
                "expected_matrix_tc": matrix_tc,
                "phi0_ok": False,
                "error": str(e),
            })
    return pd.DataFrame(rows)

sanity_df = quick_sanity_check(EMT_MODEL_SPECS)
display(sanity_df)

In [ ]:
def calibration_loss(pred: float, obs: float, mode: str) -> float:
    """Scalar loss between prediction and observation.

    mode:
      - sse: (pred-obs)^2
      - rel_sse: ((pred-obs)/obs)^2
      - log_sse: (log(pred)-log(obs))^2
    """
    if not (np.isfinite(pred) and np.isfinite(obs)):
        return 1e30
    if mode == "sse":
        return float((pred - obs) ** 2)
    if mode == "rel_sse":
        if obs == 0:
            return 1e30
        return float(((pred - obs) / obs) ** 2)
    if mode == "log_sse":
        if pred <= 0 or obs <= 0:
            return 1e30
        return float((np.log(pred) - np.log(obs)) ** 2)
    raise ValueError(f"Unknown CALIBRATION_OBJECTIVE_MODE={mode!r}")


def fit_alpha_single_point(
    *,
    phi: float,
    observed_tc: float,
    matrix_tc: float,
    fluid_tc: float,
    predict_func,
    alpha_bounds=(1e-4, 1.0),
    objective_mode: str = "log_sse",
):
    """Fit alpha for one (phi, TC) point.

    This is an *identification* step used to transfer pore-geometry information from
    dry measurements to other saturation states.
    """
    lo, hi = np.log10(alpha_bounds[0]), np.log10(alpha_bounds[1])

    def objective(log10_alpha: float) -> float:
        alpha = float(10.0 ** log10_alpha)
        try:
            pred = float(predict_func(matrix_tc, fluid_tc, float(phi), alpha))
        except Exception:
            return 1e30
        return calibration_loss(pred, float(observed_tc), objective_mode)

    res = minimize_scalar(objective, bounds=(lo, hi), method="bounded")
    if not res.success or not np.isfinite(res.x):
        return {
            "alpha_fit": np.nan,
            "fit_objective": np.nan,
            "fit_success": False,
            "fit_at_lower": False,
            "fit_at_upper": False,
        }

    alpha_fit = float(10.0 ** res.x)
    eps = 1e-3
    return {
        "alpha_fit": alpha_fit,
        "fit_objective": float(res.fun),
        "fit_success": True,
        "fit_at_lower": bool(res.x <= lo + eps),
        "fit_at_upper": bool(res.x >= hi - eps),
    }


def fit_alpha_global_state(
    *,
    phi: np.ndarray,
    observed_tc: np.ndarray,
    matrix_tc: float,
    fluid_tc: float,
    predict_func,
    alpha_bounds=(1e-4, 1.0),
    objective_mode: str = "log_sse",
):
    """Fit ONE alpha across many dry samples (robust baseline)."""
    phi = np.asarray(phi, dtype=float)
    observed_tc = np.asarray(observed_tc, dtype=float)

    mask = np.isfinite(phi) & np.isfinite(observed_tc)
    phi = phi[mask]
    observed_tc = observed_tc[mask]

    if phi.size == 0:
        return {
            "alpha_fit": np.nan,
            "fit_objective": np.nan,
            "fit_success": False,
            "fit_at_lower": False,
            "fit_at_upper": False,
        }

    lo, hi = np.log10(alpha_bounds[0]), np.log10(alpha_bounds[1])

    def objective(log10_alpha: float) -> float:
        alpha = float(10.0 ** log10_alpha)
        total = 0.0
        for p, obs in zip(phi, observed_tc):
            try:
                pred = float(predict_func(matrix_tc, fluid_tc, float(p), alpha))
            except Exception:
                return 1e30
            total += calibration_loss(pred, float(obs), objective_mode)
        return float(total)

    res = minimize_scalar(objective, bounds=(lo, hi), method="bounded")
    if not res.success or not np.isfinite(res.x):
        return {
            "alpha_fit": np.nan,
            "fit_objective": np.nan,
            "fit_success": False,
            "fit_at_lower": False,
            "fit_at_upper": False,
        }

    alpha_fit = float(10.0 ** res.x)
    eps = 1e-3
    return {
        "alpha_fit": alpha_fit,
        "fit_objective": float(res.fun),
        "fit_success": True,
        "fit_at_lower": bool(res.x <= lo + eps),
        "fit_at_upper": bool(res.x >= hi - eps),
    }


In [ ]:
def calibration_loss(pred: float, obs: float, mode: str) -> float:
    """Scalar loss between prediction and observation."""
    if not (np.isfinite(pred) and np.isfinite(obs)):
        return 1e30
    if mode == "sse":
        return float((pred - obs) ** 2)
    if mode == "rel_sse":
        if obs == 0:
            return 1e30
        return float(((pred - obs) / obs) ** 2)
    if mode == "log_sse":
        if pred <= 0 or obs <= 0:
            return 1e30
        return float((np.log(pred) - np.log(obs)) ** 2)
    raise ValueError(f"Unknown CALIBRATION_OBJECTIVE_MODE={mode!r}")


def fit_alpha_joint_sample(
    *,
    phi: float,
    observed_by_state: dict[str, float],
    matrix_tc: float,
    state_map: dict[str, tuple[str, float]],
    predict_func,
    alpha_bounds=(1e-4, 1.0),
    objective_mode: str = "log_sse",
) -> dict:
    """Fit alpha using multiple saturation states for one sample."""

    lo, hi = np.log10(alpha_bounds[0]), np.log10(alpha_bounds[1])

    def objective(log10_alpha: float) -> float:
        alpha = float(10.0 ** log10_alpha)
        total = 0.0
        for state_name, obs in observed_by_state.items():
            _, fluid_tc = state_map[state_name]
            try:
                pred = float(predict_func(matrix_tc, float(fluid_tc), float(phi), alpha))
            except Exception:
                return 1e30
            total += calibration_loss(pred, float(obs), objective_mode)
        return float(total)

    res = minimize_scalar(objective, bounds=(lo, hi), method="bounded")
    if not res.success or not np.isfinite(res.x):
        return {
            "alpha_fit": np.nan,
            "fit_objective": np.nan,
            "fit_success": False,
            "fit_at_lower": False,
            "fit_at_upper": False,
        }

    alpha_fit = float(10.0 ** res.x)
    eps = 1e-3
    return {
        "alpha_fit": alpha_fit,
        "fit_objective": float(res.fun),
        "fit_success": True,
        "fit_at_lower": bool(res.x <= lo + eps),
        "fit_at_upper": bool(res.x >= hi - eps),
    }


def fit_alpha_global_states(
    *,
    phi: np.ndarray,
    observed_by_state: dict[str, np.ndarray],
    matrix_tc: float,
    state_map: dict[str, tuple[str, float]],
    predict_func,
    alpha_bounds=(1e-4, 1.0),
    objective_mode: str = "log_sse",
) -> dict:
    """Fit ONE alpha across many samples using multiple states."""

    lo, hi = np.log10(alpha_bounds[0]), np.log10(alpha_bounds[1])

    def objective(log10_alpha: float) -> float:
        alpha = float(10.0 ** log10_alpha)
        total = 0.0
        for state_name, obs_arr in observed_by_state.items():
            _, fluid_tc = state_map[state_name]
            for p, obs in zip(phi, obs_arr):
                try:
                    pred = float(predict_func(matrix_tc, float(fluid_tc), float(p), alpha))
                except Exception:
                    return 1e30
                total += calibration_loss(pred, float(obs), objective_mode)
        return float(total)

    res = minimize_scalar(objective, bounds=(lo, hi), method="bounded")
    if not res.success or not np.isfinite(res.x):
        return {
            "alpha_fit": np.nan,
            "fit_objective": np.nan,
            "fit_success": False,
            "fit_at_lower": False,
            "fit_at_upper": False,
        }

    alpha_fit = float(10.0 ** res.x)
    eps = 1e-3
    return {
        "alpha_fit": alpha_fit,
        "fit_objective": float(res.fun),
        "fit_success": True,
        "fit_at_lower": bool(res.x <= lo + eps),
        "fit_at_upper": bool(res.x >= hi - eps),
    }


def evaluate_emt_models(
    df: pd.DataFrame,
    id_col: str,
    phi_col: str,
    state_order,
    matrix_tc: float,
    calibration_state: str = "dry",
    calibration_states: list[str] | None = None,
    alpha_bounds=(1e-4, 1.0),
    model_specs=None,
    alpha_fit_mode: str = "per_sample",  # per_sample | global
    objective_mode: str = "log_sse",
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:

    if model_specs is None:
        model_specs = EMT_MODEL_SPECS

    state_map = {state: (meas_col, fluid_tc) for state, meas_col, fluid_tc in state_order}

    if calibration_states is None:
        calibration_states = [calibration_state]

    for st in calibration_states:
        if st not in state_map:
            raise ValueError(f"Calibration state {st!r} not found in STATE_ORDER.")
        meas_col, _ = state_map[st]
        if meas_col not in df.columns:
            raise ValueError(f"Calibration measurement column {meas_col!r} not found in df.")

    predictor_rows: list[dict] = []
    calibration_rows: list[dict] = []

    # -------------------------
    # Optional global alpha per model
    # -------------------------
    global_alpha_by_model: dict[str, dict] = {}
    if alpha_fit_mode == "global":
        # Build arrays for rows that have all calibration states available
        need_cols = [phi_col] + [state_map[s][0] for s in calibration_states]
        cal_df = df[need_cols].copy().dropna()
        phi_all = cal_df[phi_col].to_numpy(dtype=float)
        observed_all = {s: cal_df[state_map[s][0]].to_numpy(dtype=float) for s in calibration_states}

        for model_name, spec in model_specs.items():
            if not spec["uses_alpha"]:
                continue
            global_alpha_by_model[model_name] = fit_alpha_global_states(
                phi=phi_all,
                observed_by_state=observed_all,
                matrix_tc=matrix_tc,
                state_map=state_map,
                predict_func=spec["func"],
                alpha_bounds=alpha_bounds,
                objective_mode=objective_mode,
            )

    elif alpha_fit_mode != "per_sample":
        raise ValueError(f"Unknown alpha_fit_mode={alpha_fit_mode!r}")

    # -------------------------
    # Per-sample predictions
    # -------------------------
    for _, row in df.iterrows():
        sample_id = row.get(id_col, None)
        phi = row.get(phi_col, np.nan)
        if pd.isna(phi):
            continue
        phi = float(phi)

        observed_by_state: dict[str, float] = {}
        for st in calibration_states:
            meas_col, _ = state_map[st]
            obs_val = row.get(meas_col, np.nan)
            if pd.isna(obs_val):
                observed_by_state = {}
                break
            observed_by_state[st] = float(obs_val)

        if not observed_by_state:
            continue

        for model_name, spec in model_specs.items():
            predict_func = spec["func"]
            uses_alpha = bool(spec["uses_alpha"])

            alpha_fit = np.nan
            fit_success = False
            fit_objective = np.nan
            fit_at_lower = False
            fit_at_upper = False
            alpha_source = None

            if uses_alpha:
                if alpha_fit_mode == "global":
                    info = global_alpha_by_model.get(model_name, {})
                    alpha_fit = float(info.get("alpha_fit", np.nan))
                    fit_success = bool(info.get("fit_success", False)) and np.isfinite(alpha_fit)
                    fit_objective = float(info.get("fit_objective", np.nan))
                    fit_at_lower = bool(info.get("fit_at_lower", False))
                    fit_at_upper = bool(info.get("fit_at_upper", False))
                    alpha_source = "global"
                else:
                    info = fit_alpha_joint_sample(
                        phi=phi,
                        observed_by_state=observed_by_state,
                        matrix_tc=matrix_tc,
                        state_map=state_map,
                        predict_func=predict_func,
                        alpha_bounds=alpha_bounds,
                        objective_mode=objective_mode,
                    )
                    alpha_fit = float(info["alpha_fit"]) if np.isfinite(info["alpha_fit"]) else np.nan
                    fit_success = bool(info["fit_success"]) and np.isfinite(alpha_fit)
                    fit_objective = float(info["fit_objective"]) if np.isfinite(info["fit_objective"]) else np.nan
                    fit_at_lower = bool(info["fit_at_lower"])
                    fit_at_upper = bool(info["fit_at_upper"])
                    alpha_source = "per_sample"
            else:
                # No alpha: require finite predictions on all calibration states
                ok = True
                total = 0.0
                for st, obs in observed_by_state.items():
                    _, fluid_tc = state_map[st]
                    try:
                        pred = float(predict_func(matrix_tc, float(fluid_tc), phi, None))
                    except Exception:
                        ok = False
                        break
                    if not np.isfinite(pred):
                        ok = False
                        break
                    total += calibration_loss(pred, obs, objective_mode)
                fit_success = ok
                fit_objective = float(total) if ok else np.nan

            # calibration diagnostics: mean abs pct error across calibration states
            cal_apes = []
            if fit_success:
                for st, obs in observed_by_state.items():
                    _, fluid_tc = state_map[st]
                    try:
                        pred = float(predict_func(matrix_tc, float(fluid_tc), phi, alpha_fit if uses_alpha else None))
                    except Exception:
                        pred = np.nan
                    ape = abs(pred - obs) / abs(obs) * 100.0 if (np.isfinite(pred) and obs != 0) else np.nan
                    cal_apes.append(ape)
            cal_mape = float(np.nanmean(cal_apes)) if cal_apes else np.nan

            calibration_rows.append({
                "sample_id": sample_id,
                "model": model_name,
                "phi": phi,
                "calibration_states": "+".join(calibration_states),
                "matrix_tc": matrix_tc,
                "alpha_fit": alpha_fit,
                "alpha_source": alpha_source,
                "alpha_fit_mode": alpha_fit_mode,
                "fit_objective": fit_objective,
                "fit_success": fit_success,
                "fit_at_lower": fit_at_lower,
                "fit_at_upper": fit_at_upper,
                "uses_alpha": uses_alpha,
                "objective_mode": objective_mode,
                "calibration_mape": cal_mape,
            })

            if not fit_success:
                continue

            for state_name, meas_col, fluid_tc in state_order:
                if meas_col not in df.columns:
                    continue
                obs_val = row.get(meas_col, np.nan)
                if pd.isna(obs_val):
                    continue
                observed = float(obs_val)

                try:
                    predicted = float(predict_func(matrix_tc, float(fluid_tc), phi, alpha_fit if uses_alpha else None))
                except Exception:
                    predicted = np.nan

                error = predicted - observed if np.isfinite(predicted) else np.nan
                abs_error = abs(error) if np.isfinite(error) else np.nan
                ape = abs_error / abs(observed) * 100.0 if (np.isfinite(abs_error) and observed != 0) else np.nan

                predictor_rows.append({
                    "sample_id": sample_id,
                    "state": state_name,
                    "measurement_col": meas_col,
                    "phi": phi,
                    "matrix_tc": matrix_tc,
                    "fluid_tc": float(fluid_tc),
                    "model": model_name,
                    "alpha_fit": alpha_fit,
                    "alpha_source": alpha_source,
                    "alpha_fit_mode": alpha_fit_mode,
                    "uses_alpha": uses_alpha,
                    "observed_tc": observed,
                    "predicted_tc": predicted,
                    "error": error,
                    "abs_error": abs_error,
                    "abs_pct_error": ape,
                    "is_calibration_state": state_name in calibration_states,
                })

    predictor_df = pd.DataFrame(predictor_rows)
    calibration_df = pd.DataFrame(calibration_rows)

    summary_df = (
        predictor_df
        .groupby(["state", "model"], dropna=False)
        .agg(
            n=("observed_tc", "size"),
            mean_obs=("observed_tc", "mean"),
            mean_pred=("predicted_tc", "mean"),
            bias=("error", "mean"),
            mae=("abs_error", "mean"),
            rmse=("error", lambda s: np.sqrt(np.nanmean(np.square(s)))),
            mape=("abs_pct_error", "mean"),
        )
        .reset_index()
        .sort_values(["state", "rmse", "mae"])
    )

    alpha_summary_df = (
        calibration_df
        .groupby("model", dropna=False)
        .agg(
            n_cal=("fit_success", "size"),
            alpha_mean=("alpha_fit", "mean"),
            alpha_median=("alpha_fit", "median"),
            alpha_min=("alpha_fit", "min"),
            alpha_max=("alpha_fit", "max"),
            frac_at_lower=("fit_at_lower", "mean"),
            frac_at_upper=("fit_at_upper", "mean"),
            cal_mape=("calibration_mape", "mean"),
        )
        .reset_index()
        .sort_values("model")
    )

    return predictor_df, calibration_df, summary_df, alpha_summary_df


In [ ]:
emt_predictor_df, emt_calibration_df, emt_summary_df, emt_alpha_summary_df = evaluate_emt_models(
    df=df,
    id_col=ID_COL,
    phi_col=PHI_COL,
    state_order=STATE_ORDER,
    matrix_tc=DEFAULT_MATRIX_TC,
    calibration_state=CALIBRATION_STATE,
    calibration_states=CALIBRATION_STATES,
    alpha_bounds=DEFAULT_ALPHA_BOUNDS,
    model_specs={k:v for k,v in EMT_MODEL_SPECS.items() if k in EMT_MODELS_TO_EVAL},
    alpha_fit_mode=ALPHA_FIT_MODE,
    objective_mode=CALIBRATION_OBJECTIVE_MODE,
)

display(emt_summary_df)
display(emt_alpha_summary_df)

# Save outputs
emt_predictor_df.to_csv(OUTDIR / "emt_predictors_detailed.csv", index=False)
emt_calibration_df.to_csv(OUTDIR / "emt_calibration_detailed.csv", index=False)
emt_summary_df.to_csv(OUTDIR / "emt_predictors_summary.csv", index=False)
emt_alpha_summary_df.to_csv(OUTDIR / "emt_alpha_summary.csv", index=False)

print(f"Saved EMT outputs to: {OUTDIR}")


In [ ]:
# Misfit for joint inversion (Oil / Brine) for selected EMT models
import numpy as np
import pandas as pd

models = [m for m in ["GSA", "SCA"] if m in emt_predictor_df["model"].unique()]
states = ["oil", "brine_avg"]

sub = emt_predictor_df[
    emt_predictor_df["model"].isin(models)
    & emt_predictor_df["state"].isin(states)
].copy()

sub["rel_err_pct"] = (sub["predicted_tc"] - sub["observed_tc"]) / sub["observed_tc"] * 100.0
sub["abs_rel_err_pct"] = sub["rel_err_pct"].abs()

misfit_joint = (
    sub.groupby(["state", "model"], dropna=False)
    .agg(
        n=("observed_tc", "size"),
        bias_pct=("rel_err_pct", "mean"),
        mape_pct=("abs_rel_err_pct", "mean"),
        mae=("abs_error", "mean"),
        rmse=("error", lambda s: float(np.sqrt(np.nanmean(np.square(s))))),
        max_ape_pct=("abs_rel_err_pct", "max"),
    )
    .reset_index()
    .sort_values(["state", "rmse"])
)

display(misfit_joint)


In [ ]:
def make_summary_by_state(summary_df: pd.DataFrame) -> pd.DataFrame:
    metrics_order = ["n", "mean_pred", "bias", "mae", "rmse", "mape"]
    state_order_out = ["dry", "brine_avg", "oil"]

    state_rename = {
        "dry": "dry",
        "brine_avg": "brine",
        "oil": "oil",
    }

    out = summary_df.copy()
    out = out[out["state"].isin(state_order_out)].copy()
    out["state"] = out["state"].map(state_rename)

    table = out.pivot_table(
        index="model",
        columns="state",
        values=metrics_order,
        aggfunc="first",
    )

    table = table.swaplevel(0, 1, axis=1)

    desired_cols = []
    for state in ["dry", "brine", "oil"]:
        for metric in metrics_order:
            desired_cols.append((state, metric))

    existing_cols = [col for col in desired_cols if col in table.columns]
    table = table.reindex(columns=pd.MultiIndex.from_tuples(existing_cols))

    for col in table.columns:
        if col[1] == "n":
            table[col] = table[col].astype("Int64")
        else:
            table[col] = table[col].round(3)

    table.columns.names = [None, None]
    return table

emt_summary_state_table = make_summary_by_state(emt_summary_df)
display(emt_summary_state_table)

In [ ]:
# Guard: this cell expects bounds/models to be defined above.
_required = ["wiener_parallel", "wiener_series", "hs_upper", "hs_lower", "EMT_MODEL_SPECS"]
_missing = [name for name in _required if name not in globals()]
if _missing:
    raise RuntimeError(
        "Missing symbols: " + ", ".join(_missing) +
        ". Run the notebook top-to-bottom (Kernel -> Restart & Run All)."
    )

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd

# -----------------------------
# Шрифт
# -----------------------------
mpl.rcParams["font.family"] = "serif"
mpl.rcParams["font.serif"] = ["Times New Roman", "Times", "DejaVu Serif"]

# -----------------------------
# Настройки
# -----------------------------
STATES_TO_PLOT = ["dry", "oil", "brine_avg"]
POROSITY_AXIS_IN_PERCENT = True
YMIN, YMAX = 0.0, 3.5
XMIN, XMAX = 0.0, 25.0

# Для каких границ проверять выход точки:
# "hs" или "wiener"
BOUND_FOR_ERROR_BARS = "wiener"

# Погрешность для "проблемных" водонасыщенных образцов
REL_ERROR_BAR = 0.025   # 2.5%

STATE_MAP = {state: (meas_col, fluid_tc) for state, meas_col, fluid_tc in STATE_ORDER}

STATE_TITLES = {
    "dry": "Dry",
    "oil": "Oil-saturated",
    "brine_avg": "Brine-saturated",
}

EMT_COLORS = {
    "MT": "red",
    "SCA": "green",
    "DEM": "blue",
    "GSA": "purple",
    "Bruggeman": "orange",
}

USE_CALIBRATED_ALPHA = True
MANUAL_ALPHA_BY_MODEL = {
    "MT": 0.10,
    "SCA": 0.10,
    "DEM": 0.10,
    "GSA": 0.10,
}

def get_phi_range(phi_max_percent=25.0, n=400):
    phi_max = phi_max_percent / 100.0
    return np.linspace(0.0, phi_max, n)

def build_alpha_map(emt_calibration_df=None):
    if USE_CALIBRATED_ALPHA and emt_calibration_df is not None and not emt_calibration_df.empty:
        alpha_map = (
            emt_calibration_df
            .groupby("model", dropna=False)["alpha_fit"]
            .median()
            .to_dict()
        )
        for model_name, alpha_val in MANUAL_ALPHA_BY_MODEL.items():
            alpha_map.setdefault(model_name, alpha_val)
        return alpha_map
    return MANUAL_ALPHA_BY_MODEL.copy()

phi_grid = get_phi_range(phi_max_percent=25.0, n=400)
alpha_map = build_alpha_map(emt_calibration_df if "emt_calibration_df" in globals() else None)

# -----------------------------
# Стиль
# -----------------------------
plt.rcParams.update({
    "font.size": 13,
    "axes.titlesize": 15,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 11,
    "mathtext.fontset": "stix",
})

fig, axes = plt.subplots(1, 3, figsize=(18, 5.8), sharey=True)

for ax, state in zip(axes, STATES_TO_PLOT):
    if state not in STATE_MAP:
        ax.set_visible(False)
        continue

    meas_col, fluid_tc = STATE_MAP[state]

    if meas_col not in df.columns:
        ax.set_visible(False)
        continue

    state_df = df[[ID_COL, PHI_COL, meas_col]].copy().dropna(subset=[PHI_COL, meas_col])
    state_df = state_df[state_df[PHI_COL] <= 0.25].copy()

    if state_df.empty:
        ax.set_visible(False)
        continue

    if POROSITY_AXIS_IN_PERCENT:
        x_obs = state_df[PHI_COL].to_numpy() * 100.0
        x_grid = phi_grid * 100.0
        xlabel = "Porosity, %"
    else:
        x_obs = state_df[PHI_COL].to_numpy()
        x_grid = phi_grid
        xlabel = "Porosity, fraction"

    y_obs = state_df[meas_col].to_numpy()

    # -------------------------
    # Границы Винера и ХШ
    # -------------------------
    wiener_upper_vals = np.array([wiener_parallel(DEFAULT_MATRIX_TC, fluid_tc, phi) for phi in phi_grid])
    wiener_lower_vals = np.array([wiener_series(DEFAULT_MATRIX_TC, fluid_tc, phi) for phi in phi_grid])

    hs_upper_vals = np.array([hs_upper(DEFAULT_MATRIX_TC, fluid_tc, phi) for phi in phi_grid])
    hs_lower_vals = np.array([hs_lower(DEFAULT_MATRIX_TC, fluid_tc, phi) for phi in phi_grid])

    ax.fill_between(
        x_grid,
        np.minimum(wiener_lower_vals, wiener_upper_vals),
        np.maximum(wiener_lower_vals, wiener_upper_vals),
        alpha=0.18,
        label="Wiener bounds",
        zorder=1,
    )

    ax.fill_between(
        x_grid,
        np.minimum(hs_lower_vals, hs_upper_vals),
        np.maximum(hs_lower_vals, hs_upper_vals),
        alpha=0.30,
        label="Hashin–Shtrikman bounds",
        zorder=2,
    )

    # -------------------------
    # Экспериментальные точки
    # -------------------------
    ax.scatter(
        x_obs,
        y_obs,
        color="black",
        alpha=0.5,
        s=36,
        label="Experimental data",
        zorder=5,
    )

    # -------------------------
    # Error bars для brine-образцов вне границ
    # -------------------------
    if state == "brine_avg":
        phi_obs = state_df[PHI_COL].to_numpy(dtype=float)

        w_lo_obs = np.array([wiener_series(DEFAULT_MATRIX_TC, fluid_tc, phi) for phi in phi_obs])
        w_hi_obs = np.array([wiener_parallel(DEFAULT_MATRIX_TC, fluid_tc, phi) for phi in phi_obs])

        hs_lo_obs = np.array([hs_lower(DEFAULT_MATRIX_TC, fluid_tc, phi) for phi in phi_obs])
        hs_hi_obs = np.array([hs_upper(DEFAULT_MATRIX_TC, fluid_tc, phi) for phi in phi_obs])

        if BOUND_FOR_ERROR_BARS == "wiener":
            lower_bound = np.minimum(w_lo_obs, w_hi_obs)
            upper_bound = np.maximum(w_lo_obs, w_hi_obs)
        else:
            lower_bound = np.minimum(hs_lo_obs, hs_hi_obs)
            upper_bound = np.maximum(hs_lo_obs, hs_hi_obs)

        outside_mask = (y_obs < lower_bound) | (y_obs > upper_bound)

        if np.any(outside_mask):
            yerr = REL_ERROR_BAR * y_obs[outside_mask]
            ax.errorbar(
                x_obs[outside_mask],
                y_obs[outside_mask],
                yerr=yerr,
                fmt="none",
                ecolor="black",
                elinewidth=1.2,
                capsize=3,
                alpha=0.9,
                zorder=6,
                label="2.5% uncertainty" if ax is axes[0] else None,
            )

    # -------------------------
    # EMT-модели
    # -------------------------
    for model_name, spec in EMT_MODEL_SPECS.items():
        func = spec["func"]
        uses_alpha = spec["uses_alpha"]
        color = EMT_COLORS.get(model_name, None)
        alpha = alpha_map.get(model_name, np.nan)

        y = []
        for phi in phi_grid:
            try:
                val = func(
                    DEFAULT_MATRIX_TC,
                    fluid_tc,
                    float(phi),
                    alpha if uses_alpha else None
                )
            except Exception:
                val = np.nan
            y.append(val)

        label = f"{model_name} ($\\alpha$={alpha:.3g})" if uses_alpha else model_name

        ax.plot(
            x_grid,
            y,
            linewidth=2.2,
            color=color,
            label=label,
            zorder=4,
        )

    ax.set_title(STATE_TITLES.get(state, state))
    ax.set_xlabel(xlabel)
    ax.set_ylim(YMIN, YMAX)
    ax.set_xlim(XMIN, XMAX)
    ax.grid(True, alpha=0.25)

axes[0].set_ylabel("Thermal conductivity, W/(m·K)")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, -0.05),
)

fig.tight_layout(rect=[0, 0.10, 1, 1])
plt.show()

#### emt sensetivity analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

In [ ]:
# -----------------------------
# Шрифт и стиль
# -----------------------------
mpl.rcParams["font.family"] = "serif"
mpl.rcParams["font.serif"] = ["Times New Roman", "Times", "DejaVu Serif"]
mpl.rcParams["mathtext.fontset"] = "stix"

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
})

# -----------------------------
# Какие состояния показывать
# -----------------------------
STATES_TO_PLOT = ["dry", "oil", "brine_avg"]

STATE_TITLES = {
    "dry": "Dry",
    "oil": "Oil-saturated",
    "brine_avg": "Brine-saturated",
}

# -----------------------------
# Aspect ratio values for sensitivity analysis
# -----------------------------
ALPHA_VALUES = np.logspace(-2, 0, 5)
# например: 0.001, 0.004, 0.016, 0.063, 0.251, 1.0

# -----------------------------
# Пористость для модельных кривых
# -----------------------------
PHI_GRID = np.linspace(0.0, 0.25, 300)

# -----------------------------
# Пределы осей
# -----------------------------
POROSITY_AXIS_IN_PERCENT = True
XMIN, XMAX = 0.0, 25.0
YMIN, YMAX = 0.0, 3.5

# -----------------------------
# Показывать ли экспериментальные точки
# -----------------------------
SHOW_EXPERIMENTAL_DATA = True
EXPERIMENTAL_ALPHA = 0.35
EXPERIMENTAL_SIZE = 22

# -----------------------------
# Цвета для разных alpha
# -----------------------------
ALPHA_CMAP = mpl.colormaps["viridis"]

In [ ]:
STATE_MAP = {state: (meas_col, fluid_tc) for state, meas_col, fluid_tc in STATE_ORDER}

In [ ]:
def format_alpha_label(alpha):
    return rf"$\alpha={alpha:.3g}$"


def predict_model_curve(model_func, matrix_tc, fluid_tc, phi_grid, alpha=None, uses_alpha=True):
    y = []

    for phi in phi_grid:
        try:
            if uses_alpha:
                val = model_func(matrix_tc, fluid_tc, float(phi), float(alpha))
            else:
                val = model_func(matrix_tc, fluid_tc, float(phi), None)
        except Exception:
            val = np.nan

        y.append(val)

    return np.asarray(y, dtype=float)


def get_state_dataframe(df, phi_col, meas_col, id_col=None):
    cols = [phi_col, meas_col]
    if id_col is not None and id_col in df.columns:
        cols = [id_col] + cols

    out = df[cols].copy().dropna(subset=[phi_col, meas_col])
    out = out[out[phi_col] <= 0.25].copy()
    return out

In [ ]:
MODEL_ORDER = ["MT", "GSA", "DEM", "SCA"]
# если у вас модель называется GSA, замените "OSP" на "GSA"

def plot_alpha_fluid_sensitivity(
    df,
    emt_model_specs,
    state_map,
    model_order,
    state_titles,
    phi_grid,
    alpha_values,
    default_matrix_tc,
    phi_col,
    id_col=None,
    porosity_axis_in_percent=True,
    show_experimental_data=True,
    experimental_alpha=0.35,
    experimental_size=22,
    xmin=0.0,
    xmax=25.0,
    ymin=0.0,
    ymax=3.5,
):
    nrows = len(model_order)
    ncols = len(STATES_TO_PLOT)

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(16, 14),
        sharex=True,
        sharey=True
    )

    if nrows == 1:
        axes = np.array([axes])

    alpha_colors = [ALPHA_CMAP(i) for i in np.linspace(0.10, 0.95, len(alpha_values))]

    for i_model, model_name in enumerate(model_order):
        if model_name not in emt_model_specs:
            for j in range(ncols):
                axes[i_model, j].set_visible(False)
            continue

        spec = emt_model_specs[model_name]
        model_func = spec["func"]
        uses_alpha = spec.get("uses_alpha", True)

        for j_state, state in enumerate(STATES_TO_PLOT):
            ax = axes[i_model, j_state]

            if state not in state_map:
                ax.set_visible(False)
                continue

            meas_col, fluid_tc = state_map[state]

            if meas_col not in df.columns:
                ax.set_visible(False)
                continue

            state_df = get_state_dataframe(df, phi_col=phi_col, meas_col=meas_col, id_col=id_col)

            if porosity_axis_in_percent:
                x_grid = phi_grid * 100.0
                xlabel = "Porosity, %"
            else:
                x_grid = phi_grid
                xlabel = "Porosity, fraction"

            # Экспериментальные точки
            if show_experimental_data and not state_df.empty:
                if porosity_axis_in_percent:
                    x_obs = state_df[phi_col].to_numpy(dtype=float) * 100.0
                else:
                    x_obs = state_df[phi_col].to_numpy(dtype=float)

                y_obs = state_df[meas_col].to_numpy(dtype=float)

                ax.scatter(
                    x_obs,
                    y_obs,
                    color="black",
                    alpha=experimental_alpha,
                    s=experimental_size,
                    zorder=2,
                    label="Experimental data" if (i_model == 0 and j_state == 0) else None,
                )

            # Семейство кривых по alpha
            for color, alpha in zip(alpha_colors, alpha_values):
                y = predict_model_curve(
                    model_func=model_func,
                    matrix_tc=default_matrix_tc,
                    fluid_tc=fluid_tc,
                    phi_grid=phi_grid,
                    alpha=alpha,
                    uses_alpha=uses_alpha,
                )

                label = format_alpha_label(alpha) if (i_model == 0 and j_state == 0) else None

                ax.plot(
                    x_grid,
                    y,
                    color=color,
                    lw=2.0,
                    zorder=3,
                    label=label,
                )

            # Заголовки только сверху
            if i_model == 0:
                ax.set_title(state_titles.get(state, state))

            # Подписи строк слева
            if j_state == 0:
                ax.set_ylabel(f"{model_name}\nThermal conductivity, W/(m·K)")
            else:
                ax.set_ylabel("")

            # Подписи x только снизу
            if i_model == nrows - 1:
                ax.set_xlabel(xlabel)

            ax.set_xlim(xmin, xmax)
            ax.set_ylim(ymin, ymax)
            ax.grid(True, alpha=0.25)

    handles, labels = axes[0, 0].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=min(len(labels), 4),
        frameon=False,
        bbox_to_anchor=(0.5, 0.02),
    )

    fig.tight_layout(rect=[0, 0.06, 1, 1])
    plt.show()

    return fig

In [ ]:
fig = plot_alpha_fluid_sensitivity(
    df=df,
    emt_model_specs=EMT_MODEL_SPECS,
    state_map=STATE_MAP,
    model_order=MODEL_ORDER,
    state_titles=STATE_TITLES,
    phi_grid=PHI_GRID,
    alpha_values=ALPHA_VALUES,
    default_matrix_tc=DEFAULT_MATRIX_TC,
    phi_col=PHI_COL,
    id_col=ID_COL,
    porosity_axis_in_percent=POROSITY_AXIS_IN_PERCENT,
    show_experimental_data=True,
    experimental_alpha=EXPERIMENTAL_ALPHA,
    experimental_size=EXPERIMENTAL_SIZE,
    xmin=XMIN,
    xmax=XMAX,
    ymin=YMIN,
    ymax=YMAX,
)

In [ ]:
def compute_alpha_sensitivity_curve(model_func, matrix_tc, fluid_tc, phi_grid, alpha_values, uses_alpha=True):
    log_alpha = np.log10(alpha_values)

    curves = []
    for alpha in alpha_values:
        y = predict_model_curve(
            model_func=model_func,
            matrix_tc=matrix_tc,
            fluid_tc=fluid_tc,
            phi_grid=phi_grid,
            alpha=alpha,
            uses_alpha=uses_alpha,
        )
        curves.append(y)

    curves = np.asarray(curves)  # shape = (n_alpha, n_phi)

    # производная по log10(alpha)
    sensitivity = np.gradient(curves, log_alpha, axis=0)

    # можно взять средний модуль по alpha
    mean_abs_sensitivity = np.nanmean(np.abs(sensitivity), axis=0)

    return mean_abs_sensitivity

In [ ]:
MODEL_COLORS = {
    "MT": "red",
    "GSA": "purple",
    "DEM": "blue",
    "SCA": "green",
}

def plot_mean_alpha_sensitivity(
    emt_model_specs,
    state_map,
    model_order,
    state_titles,
    phi_grid,
    alpha_values,
    default_matrix_tc,
    porosity_axis_in_percent=True,
):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), sharey=True)

    for ax, state in zip(axes, STATES_TO_PLOT):
        if state not in state_map:
            ax.set_visible(False)
            continue

        _, fluid_tc = state_map[state]

        for model_name in model_order:
            if model_name not in emt_model_specs:
                continue

            spec = emt_model_specs[model_name]
            model_func = spec["func"]
            uses_alpha = spec.get("uses_alpha", True)

            s = compute_alpha_sensitivity_curve(
                model_func=model_func,
                matrix_tc=default_matrix_tc,
                fluid_tc=fluid_tc,
                phi_grid=phi_grid,
                alpha_values=alpha_values,
                uses_alpha=uses_alpha,
            )

            if porosity_axis_in_percent:
                x = phi_grid * 100.0
                xlabel = "Porosity, %"
            else:
                x = phi_grid
                xlabel = "Porosity, fraction"

            ax.plot(
                x,
                s,
                lw=2.2,
                color=MODEL_COLORS.get(model_name, None),
                label=model_name,
            )

        ax.set_title(state_titles.get(state, state))
        ax.set_xlabel(xlabel)
        ax.grid(True, alpha=0.25)

    axes[0].set_ylabel(r"Mean sensitivity to $\log_{10}\alpha$, W/(m$\cdot$K)")

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=len(model_order),
        frameon=False,
        bbox_to_anchor=(0.5, -0.02),
    )

    fig.tight_layout(rect=[0, 0.06, 1, 1])
    plt.show()

    return fig

In [ ]:
fig_sens = plot_mean_alpha_sensitivity(
    emt_model_specs=EMT_MODEL_SPECS,
    state_map=STATE_MAP,
    model_order=MODEL_ORDER,
    state_titles=STATE_TITLES,
    phi_grid=PHI_GRID,
    alpha_values=ALPHA_VALUES,
    default_matrix_tc=DEFAULT_MATRIX_TC,
    porosity_axis_in_percent=POROSITY_AXIS_IN_PERCENT,
)

In [ ]:
STATES_TO_PLOT = ["dry", "oil", "brine_avg"]

STATE_TITLES = {
    "dry": "Dry",
    "oil": "Oil-saturated",
    "brine_avg": "Brine-saturated",
}

STATE_MAP = {state: (meas_col, fluid_tc) for state, meas_col, fluid_tc in STATE_ORDER}

MODEL_ORDER = ["MT", "GSA", "DEM", "SCA"]

MODEL_COLORS = {
    "MT": "red",
    "GSA": "purple",
    "DEM": "blue",
    "SCA": "green",
}

# сетка по aspect ratio
ALPHA_GRID = np.logspace(-4, 0, 80)

# сетка по пористости
PHI_GRID = np.linspace(0.0, 0.25, 250)

POROSITY_AXIS_IN_PERCENT = True
SAVE_FIGURES = False

In [ ]:
def predict_model_curve_over_alpha(model_func, matrix_tc, fluid_tc, phi, alpha_grid, uses_alpha=True):
    vals = []

    for alpha in alpha_grid:
        try:
            if uses_alpha:
                val = model_func(matrix_tc, fluid_tc, float(phi), float(alpha))
            else:
                val = model_func(matrix_tc, fluid_tc, float(phi))
        except TypeError:
            try:
                val = model_func(matrix_tc, fluid_tc, float(phi), float(alpha) if uses_alpha else None)
            except Exception:
                val = np.nan
        except Exception:
            val = np.nan

        vals.append(val)

    return np.asarray(vals, dtype=float)


def build_model_alpha_phi_matrix(model_func, matrix_tc, fluid_tc, phi_grid, alpha_grid, uses_alpha=True):
    """
    Возвращает массив shape = (n_alpha, n_phi),
    где строки соответствуют alpha, столбцы — phi.
    """
    curves = []

    for alpha in alpha_grid:
        y_phi = []
        for phi in phi_grid:
            try:
                if uses_alpha:
                    val = model_func(matrix_tc, fluid_tc, float(phi), float(alpha))
                else:
                    val = model_func(matrix_tc, fluid_tc, float(phi))
            except TypeError:
                try:
                    val = model_func(matrix_tc, fluid_tc, float(phi), float(alpha) if uses_alpha else None)
                except Exception:
                    val = np.nan
            except Exception:
                val = np.nan

            y_phi.append(val)

        curves.append(y_phi)

    return np.asarray(curves, dtype=float)


def compute_relative_sensitivity_curve(model_func, matrix_tc, fluid_tc, phi_grid, alpha_grid, uses_alpha=True):
    """
    S_rel(phi) = mean_alpha | d log10(lambda_eff) / d log10(alpha) |
    """
    curves = build_model_alpha_phi_matrix(
        model_func=model_func,
        matrix_tc=matrix_tc,
        fluid_tc=fluid_tc,
        phi_grid=phi_grid,
        alpha_grid=alpha_grid,
        uses_alpha=uses_alpha,
    )

    log_alpha = np.log10(alpha_grid)

    # только положительные значения lambda_eff допустимы для log10
    curves_safe = np.where(curves > 0.0, curves, np.nan)
    log_curves = np.log10(curves_safe)

    sensitivity = np.gradient(log_curves, log_alpha, axis=0)
    mean_abs_sensitivity = np.nanmean(np.abs(sensitivity), axis=0)

    return mean_abs_sensitivity


def compute_reduced_sensitivity_curve(model_func, matrix_tc, fluid_tc, phi_grid, alpha_grid, uses_alpha=True):
    """
    lambda_red = (lambda_eff - lambda_f) / (lambda_m - lambda_f)
    S_red(phi) = mean_alpha | d lambda_red / d log10(alpha) |
    """
    curves = build_model_alpha_phi_matrix(
        model_func=model_func,
        matrix_tc=matrix_tc,
        fluid_tc=fluid_tc,
        phi_grid=phi_grid,
        alpha_grid=alpha_grid,
        uses_alpha=uses_alpha,
    )

    denom = matrix_tc - fluid_tc
    if np.isclose(denom, 0.0):
        return np.full_like(phi_grid, np.nan, dtype=float)

    lambda_red = (curves - fluid_tc) / denom

    log_alpha = np.log10(alpha_grid)
    sensitivity = np.gradient(lambda_red, log_alpha, axis=0)
    mean_abs_sensitivity = np.nanmean(np.abs(sensitivity), axis=0)

    return mean_abs_sensitivity

In [ ]:
def plot_relative_sensitivity_by_state(
    emt_model_specs,
    state_map,
    states_to_plot,
    state_titles,
    model_order,
    model_colors,
    matrix_tc,
    phi_grid,
    alpha_grid,
    porosity_axis_in_percent=True,
    figsize=(18, 5.6),
    save_prefix=None,
):
    fig, axes = plt.subplots(1, 3, figsize=figsize, sharey=True)

    for ax, state in zip(axes, states_to_plot):
        if state not in state_map:
            ax.set_visible(False)
            continue

        _, fluid_tc = state_map[state]

        for model_name in model_order:
            if model_name not in emt_model_specs:
                continue

            spec = emt_model_specs[model_name]
            model_func = spec["func"]
            uses_alpha = spec.get("uses_alpha", True)

            s_rel = compute_relative_sensitivity_curve(
                model_func=model_func,
                matrix_tc=matrix_tc,
                fluid_tc=fluid_tc,
                phi_grid=phi_grid,
                alpha_grid=alpha_grid,
                uses_alpha=uses_alpha,
            )

            if porosity_axis_in_percent:
                x = phi_grid * 100.0
                xlabel = "Porosity, %"
            else:
                x = phi_grid
                xlabel = "Porosity, fraction"

            ax.plot(
                x,
                s_rel,
                lw=2.2,
                color=model_colors.get(model_name, None),
                label=model_name,
            )

        ax.set_title(state_titles.get(state, state))
        ax.set_xlabel(xlabel)
        ax.grid(True, alpha=0.25)

    axes[0].set_ylabel(
        r"Mean $\left|\partial \log_{10}\lambda_{\mathrm{eff}} / \partial \log_{10}\alpha\right|$"
    )

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=len(model_order),
        frameon=False,
        bbox_to_anchor=(0.5, -0.02),
    )

    fig.tight_layout(rect=[0, 0.06, 1, 1])

    if save_prefix is not None:
        fig.savefig(f"{save_prefix}.png", dpi=600, bbox_inches="tight")
        fig.savefig(f"{save_prefix}.pdf", bbox_inches="tight")

    plt.show()
    return fig

In [ ]:
def plot_reduced_sensitivity_by_state(
    emt_model_specs,
    state_map,
    states_to_plot,
    state_titles,
    model_order,
    model_colors,
    matrix_tc,
    phi_grid,
    alpha_grid,
    porosity_axis_in_percent=True,
    figsize=(18, 5.6),
    save_prefix=None,
):
    fig, axes = plt.subplots(1, 3, figsize=figsize, sharey=True)

    for ax, state in zip(axes, states_to_plot):
        if state not in state_map:
            ax.set_visible(False)
            continue

        _, fluid_tc = state_map[state]

        for model_name in model_order:
            if model_name not in emt_model_specs:
                continue

            spec = emt_model_specs[model_name]
            model_func = spec["func"]
            uses_alpha = spec.get("uses_alpha", True)

            s_red = compute_reduced_sensitivity_curve(
                model_func=model_func,
                matrix_tc=matrix_tc,
                fluid_tc=fluid_tc,
                phi_grid=phi_grid,
                alpha_grid=alpha_grid,
                uses_alpha=uses_alpha,
            )

            if porosity_axis_in_percent:
                x = phi_grid * 100.0
                xlabel = "Porosity, %"
            else:
                x = phi_grid
                xlabel = "Porosity, fraction"

            ax.plot(
                x,
                s_red,
                lw=2.2,
                color=model_colors.get(model_name, None),
                label=model_name,
            )

        ax.set_title(state_titles.get(state, state))
        ax.set_xlabel(xlabel)
        ax.grid(True, alpha=0.25)

    axes[0].set_ylabel(
        r"Mean $\left|\partial \lambda_{\mathrm{red}} / \partial \log_{10}\alpha\right|$"
    )

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=len(model_order),
        frameon=False,
        bbox_to_anchor=(0.5, -0.02),
    )

    fig.tight_layout(rect=[0, 0.06, 1, 1])

    if save_prefix is not None:
        fig.savefig(f"{save_prefix}.png", dpi=600, bbox_inches="tight")
        fig.savefig(f"{save_prefix}.pdf", bbox_inches="tight")

    plt.show()
    return fig

In [ ]:
fig_rel = plot_relative_sensitivity_by_state(
    emt_model_specs=EMT_MODEL_SPECS,
    state_map=STATE_MAP,
    states_to_plot=STATES_TO_PLOT,
    state_titles=STATE_TITLES,
    model_order=MODEL_ORDER,
    model_colors=MODEL_COLORS,
    matrix_tc=DEFAULT_MATRIX_TC,
    phi_grid=PHI_GRID,
    alpha_grid=ALPHA_GRID,
    porosity_axis_in_percent=POROSITY_AXIS_IN_PERCENT,
    save_prefix="relative_sensitivity_by_state" if SAVE_FIGURES else None,
)

fig_red = plot_reduced_sensitivity_by_state(
    emt_model_specs=EMT_MODEL_SPECS,
    state_map=STATE_MAP,
    states_to_plot=STATES_TO_PLOT,
    state_titles=STATE_TITLES,
    model_order=MODEL_ORDER,
    model_colors=MODEL_COLORS,
    matrix_tc=DEFAULT_MATRIX_TC,
    phi_grid=PHI_GRID,
    alpha_grid=ALPHA_GRID,
    porosity_axis_in_percent=POROSITY_AXIS_IN_PERCENT,
    save_prefix="reduced_sensitivity_by_state" if SAVE_FIGURES else None,
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams["font.family"] = "serif"
mpl.rcParams["font.serif"] = ["Times New Roman", "Times", "DejaVu Serif"]
mpl.rcParams["mathtext.fontset"] = "stix"

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
})

# aspect ratio grid
ALPHA_GRID = np.logspace(-4, 0, 300)

# fixed porosity for one figure
PHI_FIXED = 0.10   # например 10%

# states
STATES_TO_PLOT = ["dry", "oil", "brine_avg"]

STATE_TITLES = {
    "dry": "Dry",
    "oil": "Oil-saturated",
    "brine_avg": "Brine-saturated",
}

STATE_MAP = {state: (meas_col, fluid_tc) for state, meas_col, fluid_tc in STATE_ORDER}

# colors for models
MODEL_COLORS = {
    "MT": "red",
    "OSP": "purple",   # или "GSA", если у вас так называется модель
    "GSA": "purple",
    "DEM": "blue",
    "SCA": "green",
}

In [ ]:
def predict_vs_alpha(model_func, matrix_tc, fluid_tc, phi, alpha_grid, uses_alpha=True):
    y = []

    for alpha in alpha_grid:
        try:
            if uses_alpha:
                val = model_func(matrix_tc, fluid_tc, float(phi), float(alpha))
            else:
                val = model_func(matrix_tc, fluid_tc, float(phi))
        except TypeError:
            # на случай другой сигнатуры
            try:
                val = model_func(matrix_tc, fluid_tc, float(phi), float(alpha) if uses_alpha else None)
            except Exception:
                val = np.nan
        except Exception:
            val = np.nan

        y.append(val)

    return np.asarray(y, dtype=float)

In [ ]:
def plot_models_vs_alpha_for_fixed_phi(
    emt_model_specs,
    state_map,
    states_to_plot,
    state_titles,
    matrix_tc,
    phi_fixed,
    alpha_grid,
    figsize=(16, 5.5),
    ymin=None,
    ymax=None,
    save_prefix=None,
):
    fig, axes = plt.subplots(1, 3, figsize=figsize, sharey=True)

    for ax, state in zip(axes, states_to_plot):
        if state not in state_map:
            ax.set_visible(False)
            continue

        _, fluid_tc = state_map[state]

        for model_name, spec in emt_model_specs.items():
            model_func = spec["func"]
            uses_alpha = spec.get("uses_alpha", True)

            y = predict_vs_alpha(
                model_func=model_func,
                matrix_tc=matrix_tc,
                fluid_tc=fluid_tc,
                phi=phi_fixed,
                alpha_grid=alpha_grid,
                uses_alpha=uses_alpha,
            )

            ax.plot(
                alpha_grid,
                y,
                lw=2.3,
                color=MODEL_COLORS.get(model_name, None),
                label=model_name,
            )

        ax.set_xscale("log")
        ax.set_title(state_titles.get(state, state))
        ax.set_xlabel(r"Aspect ratio $\alpha$")
        ax.grid(True, alpha=0.25)

        if ymin is not None or ymax is not None:
            ax.set_ylim(ymin, ymax)

    axes[0].set_ylabel(r"Thermal conductivity, W m$^{-1}$ K$^{-1}$")

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=len(emt_model_specs),
        frameon=False,
        bbox_to_anchor=(0.5, -0.02),
    )

    fig.suptitle(
        rf"Model sensitivity to aspect ratio at fixed porosity $\phi={phi_fixed*100:.0f}\%$",
        fontsize=15,
    )

    fig.tight_layout(rect=[0, 0.06, 1, 0.95])

    if save_prefix is not None:
        fig.savefig(f"{save_prefix}.png", dpi=600, bbox_inches="tight")
        fig.savefig(f"{save_prefix}.pdf", bbox_inches="tight")

    plt.show()
    return fig

In [ ]:
PHI_VALUES = [0.04, 0.10, 0.24]

for phi_val in PHI_VALUES:
    fig = plot_models_vs_alpha_for_fixed_phi(
        emt_model_specs=EMT_MODEL_SPECS,
        state_map=STATE_MAP,
        states_to_plot=STATES_TO_PLOT,
        state_titles=STATE_TITLES,
        matrix_tc=DEFAULT_MATRIX_TC,
        phi_fixed=phi_val,
        alpha_grid=ALPHA_GRID,
        ymin=0.0,
        ymax=3.5,
        save_prefix=f"models_vs_alpha_phi{int(phi_val*100):02d}",
    )

## Калибровка EMT-моделей по сухому состоянию и перенос на другие насыщения

На данном этапе для каждого образца рассматривается двухфазная система:

- твердая матрица с теплопроводностью $\lambda_m$,
- поровое пространство, заполненное флюидом с теплопроводностью $\lambda_f$,
- пористость $\phi$,
- эффективный микроструктурный параметр $\alpha$, интерпретируемый как параметр формы пустотного пространства.

Для каждой EMT-модели используется обобщённая функциональная запись

$$
\lambda_{\mathrm{eff}} = F_m(\lambda_m,\lambda_f,\phi,\alpha),
$$

где $m$ — индекс модели (например, MT, SCA, DEM, GSA).

### 1. Калибровка по сухому состоянию

Для каждого образца $j$ эффективный параметр микроструктуры $\alpha_j$ определяется из условия наилучшего согласования модели с измеренной теплопроводностью сухого образца:

$$
\hat{\alpha}_j^{(m)}
=
\operatorname*{arg\,min}_{\alpha \in [\alpha_{\min},\alpha_{\max}]}
\left|
F_m(\lambda_m,\lambda_{\mathrm{air}},\phi_j,\alpha)
-
\lambda_{j,\mathrm{dry}}^{\mathrm{obs}}
\right|.
$$

Здесь:

- $\lambda_{\mathrm{air}}$ — теплопроводность воздуха,
- $\lambda_{j,\mathrm{dry}}^{\mathrm{obs}}$ — измеренная теплопроводность сухого образца,
- $\hat{\alpha}_j^{(m)}$ — инвертированный микроструктурный параметр для модели $m$.

Таким образом, сухое состояние используется как **калибровочное**, поскольку оно наиболее чувствительно к геометрии пустотного пространства.

### 2. Прогноз для нефтенасыщенного и водонасыщенного состояний

После определения $\hat{\alpha}_j^{(m)}$ этот параметр фиксируется, и выполняется прямой прогноз для других типов насыщения:

$$
\lambda_{j,\mathrm{oil}}^{\mathrm{pred},(m)}
=
F_m(\lambda_m,\lambda_{\mathrm{oil}},\phi_j,\hat{\alpha}_j^{(m)}),
$$

$$
\lambda_{j,\mathrm{brine}}^{\mathrm{pred},(m)}
=
F_m(\lambda_m,\lambda_{\mathrm{brine}},\phi_j,\hat{\alpha}_j^{(m)}).
$$

Иными словами, при переходе между насыщениями предполагается, что:

- пористость $\phi_j$ не меняется,
- параметр микроструктуры $\hat{\alpha}_j^{(m)}$ не меняется,
- меняется только теплопроводность порового флюида $\lambda_f$.

Такая постановка соответствует физике задачи флюидозамещения: одна и та же структура порового пространства переносится между различными флюидами.

### 3. Ошибка прогноза

Для каждого состояния вычисляется ошибка прогноза:

$$
e_{j,s}^{(m)}
=
\lambda_{j,s}^{\mathrm{pred},(m)}
-
\lambda_{j,s}^{\mathrm{obs}},
$$

где $s \in \{\mathrm{dry},\mathrm{oil},\mathrm{brine}\}$.

Абсолютная процентная ошибка определяется как

$$
\mathrm{APE}_{j,s}^{(m)}
=
\frac{\left|e_{j,s}^{(m)}\right|}{\lambda_{j,s}^{\mathrm{obs}}}\cdot 100\%.
$$

### 4. Итоговые метрики качества

Для сравнения моделей используются следующие сводные характеристики.

Средняя абсолютная процентная ошибка:

$$
\mathrm{MAPE}^{(m)}
=
\frac{1}{N}
\sum_{j,s}
\mathrm{APE}_{j,s}^{(m)}.
$$

Среднеквадратичная ошибка:

$$
\mathrm{RMSE}^{(m)}
=
\sqrt{
\frac{1}{N}
\sum_{j,s}
\left(e_{j,s}^{(m)}\right)^2
}.
$$

Смещённость прогноза:

$$
\mathrm{Bias}^{(m)}
=
\frac{1}{N}
\sum_{j,s}
\frac{
\lambda_{j,s}^{\mathrm{pred},(m)}
-
\lambda_{j,s}^{\mathrm{obs}}
}{
\lambda_{j,s}^{\mathrm{obs}}
}
\cdot 100\%.
$$

Максимальная абсолютная процентная ошибка:

$$
\mathrm{MaxAE}^{(m)}
=
\max_{j,s}
\mathrm{APE}_{j,s}^{(m)}.
$$

### 5. Устойчивость инверсии микроструктурного параметра

Для моделей, в которых явно используется параметр $\alpha$, дополнительно оценивается ширина 95\%-го интервала:

$$
w_{95}(\alpha)
=
\alpha_{97.5} - \alpha_{2.5},
$$

где $\alpha_{2.5}$ и $\alpha_{97.5}$ — соответственно 2.5-й и 97.5-й процентили распределения инвертированных значений $\alpha$ по выборке образцов.

Чем меньше $w_{95}(\alpha)$, тем более устойчивой является инверсия микроструктурного параметра.

### 6. Особый случай модели Bruggeman

Для модели Bruggeman эффективная теплопроводность определяется без параметра формы пустотного пространства:

$$
\sum_i
\phi_i
\frac{\lambda_i-\lambda_{\mathrm{eff}}}
{\lambda_i+2\lambda_{\mathrm{eff}}}
=0.
$$

Поэтому для этой модели параметр $\alpha$ не инвертируется, а величина $w_{95}(\alpha)$ не определяется.

In [ ]:
# Да — вот **чистый рабочий код одной ячейкой** для Colab.

# Он строит **анализ чувствительности / identifiability** для EMT-моделей:

# * `MT`
# * `SCA`
# * `DEM`
# * `GSA`

# по одному выбранному образцу.

# ```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================================================
# Настройки
# =========================================================

# Ожидается, что df уже есть в памяти
# и содержит колонки:
#   Sample, porosity, tc_air, tc_oil, tc_brine_avg
#
# Ожидается, что уже определены функции:
#   mt_predict(matrix_tc, fluid_tc, phi, alpha)
#   sca_predict(matrix_tc, fluid_tc, phi, alpha)
#   dem_predict(matrix_tc, fluid_tc, phi, alpha)
#   gsa_predict(matrix_tc, fluid_tc, phi, alpha)

DEFAULT_MATRIX_TC = 2.98
DEFAULT_AIR_TC = 0.025
DEFAULT_OIL_TC = 0.13
DEFAULT_BRINE_AVG_TC = 0.60

SAMPLE_ID_COL = "Sample"
POROSITY_COL = "Porosity,%"

MEASURED_COLS = {
    "dry": "TC air",
    "brine": "TC 60",
    "oil": "TC oil",
}

FLUID_TCS = {
    "dry": DEFAULT_AIR_TC,
    "brine": DEFAULT_BRINE_AVG_TC,
    "oil": DEFAULT_OIL_TC,
}




# Выбери образец:
SAMPLE_ID = df[SAMPLE_ID_COL].iloc[0]   # можно заменить на конкретный sample id

# Сетка по aspect ratio
ALPHA_GRID = np.logspace(-4, 1, 400)

# Если True, misfit = ((pred - obs)/obs)^2
# Если False, misfit = (pred - obs)^2
NORMALIZE_BY_OBS = True

WEIGHTS = {
    "dry": 1.0,
    "brine": 1.0,
    "oil": 1.0,
}

# =========================================================
# Шрифт и стиль
# =========================================================
mpl.rcParams["font.family"] = "serif"
mpl.rcParams["font.serif"] = ["Times New Roman", "Times", "DejaVu Serif"]
mpl.rcParams["mathtext.fontset"] = "stix"

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
})

# =========================================================
# EMT модели
# =========================================================
MODEL_FUNCS = {
    "MT": mt_predict,
    "SCA": sca_predict,
    "DEM": dem_predict,
    "GSA": gsa_predict,
}

# =========================================================
# Выбор образца
# =========================================================
sample_df = df.loc[df[SAMPLE_ID_COL] == SAMPLE_ID]
if sample_df.empty:
    raise ValueError(f"Sample {SAMPLE_ID!r} not found in df")

sample_row = sample_df.iloc[0]
phi_sample = float(sample_row[POROSITY_COL])

obs = {
    "dry": float(sample_row[MEASURED_COLS["dry"]]),
    "brine": float(sample_row[MEASURED_COLS["brine"]]),
    "oil": float(sample_row[MEASURED_COLS["oil"]]),
}

# =========================================================
# Функции misfit
# =========================================================
def single_state_misfit(pred, obs_value, weight=1.0, normalize_by_obs=True):
    if normalize_by_obs:
        resid = (pred - obs_value) / obs_value
    else:
        resid = pred - obs_value
    return float(weight * resid**2)


def compute_identifiability_curves(model_func, phi, obs_dict, alpha_grid):
    dry_curve = []
    brine_curve = []
    oil_curve = []
    joint_curve = []

    for alpha in alpha_grid:
        pred_dry = model_func(DEFAULT_MATRIX_TC, FLUID_TCS["dry"], phi, float(alpha))
        pred_brine = model_func(DEFAULT_MATRIX_TC, FLUID_TCS["brine"], phi, float(alpha))
        pred_oil = model_func(DEFAULT_MATRIX_TC, FLUID_TCS["oil"], phi, float(alpha))

        md = single_state_misfit(
            pred_dry,
            obs_dict["dry"],
            weight=WEIGHTS["dry"],
            normalize_by_obs=NORMALIZE_BY_OBS,
        )
        mb = single_state_misfit(
            pred_brine,
            obs_dict["brine"],
            weight=WEIGHTS["brine"],
            normalize_by_obs=NORMALIZE_BY_OBS,
        )
        mo = single_state_misfit(
            pred_oil,
            obs_dict["oil"],
            weight=WEIGHTS["oil"],
            normalize_by_obs=NORMALIZE_BY_OBS,
        )

        dry_curve.append(md)
        brine_curve.append(mb)
        oil_curve.append(mo)
        joint_curve.append(md + mb + mo)

    return {
        "dry": np.array(dry_curve),
        "brine": np.array(brine_curve),
        "oil": np.array(oil_curve),
        "joint": np.array(joint_curve),
    }

# =========================================================
# Построение графиков
# =========================================================
panel_order = ["MT", "SCA", "DEM", "GSA"]

ncols = 2
nrows = 2
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 9), sharex=True)
axes = axes.ravel()

for ax, model_name in zip(axes, panel_order):
    model_func = MODEL_FUNCS[model_name]

    curves = compute_identifiability_curves(
        model_func=model_func,
        phi=phi_sample,
        obs_dict=obs,
        alpha_grid=ALPHA_GRID,
    )

    best_dry_alpha = ALPHA_GRID[np.argmin(curves["dry"])]
    best_brine_alpha = ALPHA_GRID[np.argmin(curves["brine"])]
    best_oil_alpha = ALPHA_GRID[np.argmin(curves["oil"])]
    best_joint_alpha = ALPHA_GRID[np.argmin(curves["joint"])]

    ax.plot(
        ALPHA_GRID,
        curves["dry"],
        lw=2.2,
        label=f"Dry only (best = {best_dry_alpha:.2e})",
    )
    ax.plot(
        ALPHA_GRID,
        curves["brine"],
        lw=2.2,
        label=f"Brine only (best = {best_brine_alpha:.2e})",
    )
    ax.plot(
        ALPHA_GRID,
        curves["oil"],
        lw=2.2,
        label=f"Oil only (best = {best_oil_alpha:.2e})",
    )
    ax.plot(
        ALPHA_GRID,
        curves["joint"],
        lw=2.8,
        label=f"Joint dry+brine+oil (best = {best_joint_alpha:.2e})",
    )

    ax.axvline(
        best_joint_alpha,
        linestyle="--",
        linewidth=2.0,
        label=rf"joint-best $\alpha$ = {best_joint_alpha:.2e}",
    )

    ax.set_xscale("log")
    ax.set_title(model_name)
    ax.set_xlabel(r"Aspect ratio $\alpha$")
    ax.set_ylabel("Weighted least-squares misfit")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(frameon=False, loc="upper right")

fig.suptitle(
    "Identifiability of effective aspect ratio from thermal conductivity data\n"
    f"Sample = {SAMPLE_ID}, porosity = {phi_sample:.3f}",
    y=0.98,
    fontsize=18,
)

fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


# fig.savefig("emt_identifiability_by_model.png", dpi=600, bbox_inches="tight")
# fig.savefig("emt_identifiability_by_model.pdf", bbox_inches="tight")


In [ ]:
!pip -q install numpy pandas scipy openpyxl tabulate

#### maps of gsa_identifiability_by_fluis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import pickle

from pathlib import Path
from matplotlib.lines import Line2D

In [ ]:
# =========================================================
# Ожидается, что уже есть:
# - df
# - gsa_predict(matrix_tc, fluid_tc, phi, alpha)
# =========================================================

# -----------------------------
# Значения теплопроводности флюидов
# -----------------------------
DEFAULT_AIR_TC = 0.025
DEFAULT_OIL_TC = 0.13
DEFAULT_BRINE_AVG_TC = 0.60

# -----------------------------
# Имена столбцов в df
# -----------------------------
SAMPLE_ID_COL = "Sample"
POROSITY_COL = "Porosity,%"

MEASURED_COLS = {
    "dry": "TC air",
    "oil": "TC oil",
    "brine": "TC 60",
}

# -----------------------------
# Теплопроводности флюидов
# -----------------------------
FLUID_TCS = {
    "dry": DEFAULT_AIR_TC,
    "oil": DEFAULT_OIL_TC,
    "brine": DEFAULT_BRINE_AVG_TC,
}

# -----------------------------
# Выбор образца
# -----------------------------
# SAMPLE_ID = df[SAMPLE_ID_COL].iloc[14]

SAMPLE_ID = df[SAMPLE_ID_COL].iloc[30]

# -----------------------------
# Сетки параметров
# -----------------------------
ALPHA_GRID = np.logspace(-4, 0, 220)
LAMBDA_M_GRID = np.linspace(2.4, 3.6, 220)

# -----------------------------
# Нормировка невязки
# -----------------------------
NORMALIZE_BY_OBS = True

# -----------------------------
# Все комбинации данных
# -----------------------------
SURFACES = {
    "Dry": ["dry"],
    "Oil": ["oil"],
    "Brine": ["brine"],
    "Dry + Oil": ["dry", "oil"],
    "Dry + Brine": ["dry", "brine"],
    "Oil + Brine": ["oil", "brine"],
    "Dry + Oil + Brine": ["dry", "oil", "brine"],
}

In [ ]:
CACHE_DIR = Path("gsa_cache")
CACHE_DIR.mkdir(exist_ok=True)

In [ ]:
def make_safe_sample_id(sample_id):
    return str(sample_id).replace("/", "_").replace("\\", "_").replace(" ", "_")


def get_cache_path(sample_id, alpha_grid=None, lambda_m_grid=None):
    safe_id = make_safe_sample_id(sample_id)

    if alpha_grid is not None and lambda_m_grid is not None:
        return CACHE_DIR / f"gsa_identifiability_{safe_id}_na{len(alpha_grid)}_nl{len(lambda_m_grid)}.pkl"

    return CACHE_DIR / f"gsa_identifiability_{safe_id}.pkl"

In [ ]:
def single_state_misfit(pred, obs_value, normalize_by_obs=True):
    if normalize_by_obs:
        resid = (pred - obs_value) / obs_value
    else:
        resid = pred - obs_value
    return float(resid**2)

In [ ]:
def compute_surface(states, alpha_grid, lambda_m_grid, phi, obs_dict, fluid_tcs, normalize_by_obs=True):
    Z = np.full((len(lambda_m_grid), len(alpha_grid)), np.nan, dtype=float)

    for i, lambda_m in enumerate(lambda_m_grid):
        for j, alpha in enumerate(alpha_grid):
            total = 0.0

            for state in states:
                pred = gsa_predict(
                    lambda_m,
                    fluid_tcs[state],
                    phi,
                    float(alpha),
                )

                total += single_state_misfit(
                    pred,
                    obs_dict[state],
                    normalize_by_obs=normalize_by_obs,
                )

            Z[i, j] = total

    return Z

In [ ]:
def compute_and_cache_identifiability_maps(
    df,
    sample_id,
    sample_id_col,
    porosity_col,
    measured_cols,
    fluid_tcs,
    alpha_grid,
    lambda_m_grid,
    surfaces,
    normalize_by_obs=True,
    force_recompute=False,
):
    cache_path = get_cache_path(sample_id, alpha_grid, lambda_m_grid)

    if cache_path.exists() and not force_recompute:
        print(f"Loading cached results from: {cache_path}")
        with open(cache_path, "rb") as f:
            payload = pickle.load(f)
        return payload

    print(f"Computing results for sample: {sample_id}")

    sample_df = df.loc[df[sample_id_col] == sample_id]
    if sample_df.empty:
        raise ValueError(f"Sample {sample_id!r} not found in df")

    sample_row = sample_df.iloc[0]

    phi_sample = float(sample_row[porosity_col])
    if phi_sample > 1.0:
        phi_sample = phi_sample / 100.0

    obs = {
        "dry": float(sample_row[measured_cols["dry"]]),
        "oil": float(sample_row[measured_cols["oil"]]),
        "brine": float(sample_row[measured_cols["brine"]]),
    }

    surface_results = {}

    for title, states in surfaces.items():
        print(f"  -> computing surface: {title}")

        J = compute_surface(
            states=states,
            alpha_grid=alpha_grid,
            lambda_m_grid=lambda_m_grid,
            phi=phi_sample,
            obs_dict=obs,
            fluid_tcs=fluid_tcs,
            normalize_by_obs=normalize_by_obs,
        )

        Jmin = np.nanmin(J)
        dJ = J - Jmin

        min_idx = np.unravel_index(np.nanargmin(J), J.shape)
        best_lambda_m = float(lambda_m_grid[min_idx[0]])
        best_alpha = float(alpha_grid[min_idx[1]])

        surface_results[title] = {
            "states": states,
            "J": J,
            "dJ": dJ,
            "best_alpha": best_alpha,
            "best_lambda_m": best_lambda_m,
        }

    payload = {
        "sample_id": sample_id,
        "phi_sample": phi_sample,
        "obs": obs,
        "alpha_grid": np.array(alpha_grid, dtype=float),
        "lambda_m_grid": np.array(lambda_m_grid, dtype=float),
        "fluid_tcs": dict(fluid_tcs),
        "surfaces": dict(surfaces),
        "normalize_by_obs": normalize_by_obs,
        "surface_results": surface_results,
    }

    with open(cache_path, "wb") as f:
        pickle.dump(payload, f)

    print(f"Saved cache to: {cache_path}")

    return payload

In [ ]:
def load_cached_identifiability_maps(sample_id, alpha_grid=None, lambda_m_grid=None):
    cache_path = get_cache_path(sample_id, alpha_grid, lambda_m_grid)

    if not cache_path.exists():
        raise FileNotFoundError(f"Cache file not found: {cache_path}")

    with open(cache_path, "rb") as f:
        payload = pickle.load(f)

    print(f"Loaded cache from: {cache_path}")
    return payload

In [ ]:
def plot_identifiability_maps_from_cache(
    payload,
    use_log_color_scale=True,
    acceptable_dj_levels=(0.01, 0.03, 0.05, 0.10),
    cmap_name="YlGnBu_r",
    figsize=(17.0, 8.8),
    save_prefix="gsa_identifiability_all_combinations_with_acceptable_contours",
):
    alpha_grid = payload["alpha_grid"]
    lambda_m_grid = payload["lambda_m_grid"]
    phi_sample = payload["phi_sample"]
    sample_id = payload["sample_id"]
    surface_results = payload["surface_results"]

    mpl.rcParams["font.family"] = "serif"
    mpl.rcParams["font.serif"] = ["Times New Roman", "Times", "DejaVu Serif"]
    mpl.rcParams["mathtext.fontset"] = "stix"

    plt.rcParams.update({
        "font.size": 12,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 9,
    })

    cmap = mpl.colormaps[cmap_name]

    # формируем plotZ только на этапе визуализации
    plot_results = {}
    for title, result in surface_results.items():
        dJ = result["dJ"]

        if use_log_color_scale:
            plotZ = np.log10(1.0 + dJ)
            colorbar_label = r"$\log_{10}(1+\Delta J)$, where $\Delta J = J(\alpha,\lambda_m)-J_{\min}$"
        else:
            plotZ = dJ
            colorbar_label = r"$\Delta J = J(\alpha,\lambda_m)-J_{\min}$"

        plot_results[title] = {
            **result,
            "plotZ": plotZ,
        }

    # общая цветовая шкала
    all_values = np.concatenate([result["plotZ"].ravel() for result in plot_results.values()])
    all_values = all_values[np.isfinite(all_values)]

    global_min = np.min(all_values)
    global_max = np.max(all_values)

    levels_fill = np.linspace(global_min, global_max, 24)
    levels_line = np.linspace(global_min, global_max, 10)

    # фигура
    fig, axes = plt.subplots(2, 4, figsize=figsize, sharex=True, sharey=True)
    axes = axes.ravel()

    fig.subplots_adjust(left=0.07, right=0.86, bottom=0.08, top=0.90, wspace=0.18, hspace=0.22)

    surface_items = list(plot_results.items())
    cf = None

    for ax, (title, result) in zip(axes, surface_items):
        plotZ = result["plotZ"]
        dJ = result["dJ"]

        # фоновая карта
        cf = ax.contourf(
            alpha_grid,
            lambda_m_grid,
            plotZ,
            levels=levels_fill,
            cmap=cmap,
        )

        # тонкие контуры по plotZ
        cs = ax.contour(
            alpha_grid,
            lambda_m_grid,
            plotZ,
            levels=levels_line,
            colors="k",
            linewidths=0.45,
            alpha=0.25,
        )
        ax.clabel(cs, inline=True, fontsize=6.5, fmt="%.2f")

        # одинаковые acceptable contours по dJ
        cs_acc = ax.contour(
            alpha_grid,
            lambda_m_grid,
            dJ,
            levels=acceptable_dj_levels,
            colors=["red", "orange", "gold", "limegreen"],
            linewidths=1.3,
        )
        ax.clabel(cs_acc, inline=True, fontsize=7, fmt="%.2f")

        # лучшая точка
        ax.plot(
            result["best_alpha"],
            result["best_lambda_m"],
            marker="*",
            markersize=11,
            color="white",
            markeredgecolor="black",
            linestyle="None",
            zorder=6,
        )

        ax.set_xscale("log")
        ax.set_title(title)
        ax.grid(True, alpha=0.18)

    # скрыть лишнюю 8-ю панель
    for k in range(len(surface_items), len(axes)):
        axes[k].set_visible(False)

    # подписи осей
    for ax in axes[4:7]:
        ax.set_xlabel(r"Aspect ratio $\alpha$")

    for ax in [axes[0], axes[4]]:
        ax.set_ylabel(r"Matrix thermal conductivity $\lambda_m$ (W m$^{-1}$ K$^{-1}$)")

    # colorbar
    cax = fig.add_axes([0.89, 0.16, 0.018, 0.66])
    cbar = fig.colorbar(cf, cax=cax)
    cbar.set_label(colorbar_label, rotation=90)

    # легенда
    legend_lines = [
        Line2D([0], [0], color="red", lw=1.5, label=r"$\Delta J = 0.01$"),
        Line2D([0], [0], color="orange", lw=1.5, label=r"$\Delta J = 0.03$"),
        Line2D([0], [0], color="gold", lw=1.5, label=r"$\Delta J = 0.05$"),
        Line2D([0], [0], color="limegreen", lw=1.5, label=r"$\Delta J = 0.10$"),
        Line2D([0], [0], marker="*", color="w", markerfacecolor="white",
               markeredgecolor="black", markersize=10, lw=0, label="Best fit"),
    ]

    fig.legend(
        handles=legend_lines,
        loc="lower center",
        ncol=5,
        frameon=False,
        bbox_to_anchor=(0.47, -0.02),
    )


    # fig.suptitle(
    #     "OSP/GSA identifiability maps for different fluid-data combinations\n"
    #     f"Sample = {sample_id}, porosity = {phi_sample:.3f}",
    #     fontsize=16,
    # )

    fig.suptitle(
        f"Porosity = {phi_sample*100:.2f}%",
        fontsize=16,
    )


    plt.show()

    fig.savefig(f"{save_prefix}.png", dpi=600, bbox_inches="tight")
    fig.savefig(f"{save_prefix}.pdf", bbox_inches="tight")

    return fig

In [ ]:
payload = compute_and_cache_identifiability_maps(
    df=df,
    sample_id=SAMPLE_ID,
    sample_id_col=SAMPLE_ID_COL,
    porosity_col=POROSITY_COL,
    measured_cols=MEASURED_COLS,
    fluid_tcs=FLUID_TCS,
    alpha_grid=ALPHA_GRID,
    lambda_m_grid=LAMBDA_M_GRID,
    surfaces=SURFACES,
    normalize_by_obs=NORMALIZE_BY_OBS,
    force_recompute=False,
)

# payload = load_cached_identifiability_maps(
#     sample_id="Sample_15",
#     alpha_grid=ALPHA_GRID,
#     lambda_m_grid=LAMBDA_M_GRID,
# )


In [ ]:
fig = plot_identifiability_maps_from_cache(
    payload,
    use_log_color_scale=True,
    acceptable_dj_levels=(0.01, 0.03, 0.05, 0.10),
    cmap_name="YlGnBu_r",
    save_prefix="gsa_identifiability_all_combinations_with_acceptable_contours",
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================================================
# Ожидается, что уже есть:
# - df
# - gsa_predict(matrix_tc, fluid_tc, phi, alpha)
# =========================================================

# -----------------------------
# Настройки
# -----------------------------
DEFAULT_MATRIX_TC = 2.98
DEFAULT_AIR_TC = 0.025
DEFAULT_OIL_TC = 0.13
DEFAULT_BRINE_AVG_TC = 0.60

SAMPLE_ID_COL = "Sample"
POROSITY_COL = "Porosity,%"

MEASURED_COLS = {
    "dry": "TC air",
    "oil": "TC oil",
    "brine": "TC 60",
}

FLUID_TCS = {
    "dry": DEFAULT_AIR_TC,
    "oil": DEFAULT_OIL_TC,
    "brine": DEFAULT_BRINE_AVG_TC,
}

SAMPLE_ID = df[SAMPLE_ID_COL].iloc[14]

# Сетки параметров
ALPHA_GRID = np.logspace(-4, 0, 220)

# Варьируем пористость в окрестности образца
PHI_RANGE_ABS = 0.06   # +/- 0.06 по абсолютной пористости
PHI_MIN_GLOBAL = 0.01
PHI_MAX_GLOBAL = 0.30
N_PHI = 220

NORMALIZE_BY_OBS = True
USE_LOG_COLOR_SCALE = True

# -----------------------------
# Стиль
# -----------------------------
mpl.rcParams["font.family"] = "serif"
mpl.rcParams["font.serif"] = ["Times New Roman", "Times", "DejaVu Serif"]
mpl.rcParams["mathtext.fontset"] = "stix"

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
})

cmap = mpl.colormaps["YlGnBu_r"]

# -----------------------------
# Данные образца
# -----------------------------
sample_df = df.loc[df[SAMPLE_ID_COL] == SAMPLE_ID]
if sample_df.empty:
    raise ValueError(f"Sample {SAMPLE_ID!r} not found in df")

sample_row = sample_df.iloc[0]

phi_sample = float(sample_row[POROSITY_COL])
if phi_sample > 1.0:
    phi_sample = phi_sample / 100.0

obs = {
    "dry": float(sample_row[MEASURED_COLS["dry"]]),
    "oil": float(sample_row[MEASURED_COLS["oil"]]),
    "brine": float(sample_row[MEASURED_COLS["brine"]]),
}

phi_min = max(PHI_MIN_GLOBAL, phi_sample - PHI_RANGE_ABS)
phi_max = min(PHI_MAX_GLOBAL, phi_sample + PHI_RANGE_ABS)
PHI_GRID = np.linspace(phi_min, phi_max, N_PHI)

# -----------------------------
# Невязка
# -----------------------------
def single_state_misfit(pred, obs_value, normalize_by_obs=True):
    if normalize_by_obs:
        resid = (pred - obs_value) / obs_value
    else:
        resid = pred - obs_value
    return float(resid**2)

def compute_surface_phi_alpha(states, alpha_grid, phi_grid, lambda_m, obs_dict):
    Z = np.full((len(phi_grid), len(alpha_grid)), np.nan, dtype=float)

    for i, phi in enumerate(phi_grid):
        for j, alpha in enumerate(alpha_grid):
            total = 0.0
            for state in states:
                pred = gsa_predict(
                    lambda_m,
                    FLUID_TCS[state],
                    float(phi),
                    float(alpha),
                )
                total += single_state_misfit(
                    pred,
                    obs_dict[state],
                    normalize_by_obs=NORMALIZE_BY_OBS,
                )
            Z[i, j] = total
    return Z

SURFACES = {
    "Dry": ["dry"],
    "Oil": ["oil"],
    "Brine": ["brine"],
    "Joint dry + oil + brine": ["dry", "oil", "brine"],
}

surface_results = {}

for title, states in SURFACES.items():
    J = compute_surface_phi_alpha(
        states=states,
        alpha_grid=ALPHA_GRID,
        phi_grid=PHI_GRID,
        lambda_m=DEFAULT_MATRIX_TC,
        obs_dict=obs,
    )

    Jmin = np.nanmin(J)
    dJ = J - Jmin

    if USE_LOG_COLOR_SCALE:
        plotZ = np.log10(1.0 + dJ)
        colorbar_label = r"$\log_{10}(1+\Delta J)$, where $\Delta J = J(\phi,\alpha)-J_{\min}$"
    else:
        plotZ = dJ
        colorbar_label = r"$\Delta J = J(\phi,\alpha)-J_{\min}$"

    min_idx = np.unravel_index(np.nanargmin(J), J.shape)
    best_phi = PHI_GRID[min_idx[0]]
    best_alpha = ALPHA_GRID[min_idx[1]]

    surface_results[title] = {
        "plotZ": plotZ,
        "best_phi": best_phi,
        "best_alpha": best_alpha,
    }

# -----------------------------
# Построение
# -----------------------------
fig, axes = plt.subplots(2, 2, figsize=(13.5, 10.0), sharex=True, sharey=True)
axes = axes.ravel()

fig.subplots_adjust(left=0.08, right=0.86, bottom=0.08, top=0.92, wspace=0.18, hspace=0.22)

cf = None

for ax, (title, result) in zip(axes, surface_results.items()):
    plotZ = result["plotZ"]

    levels_fill = np.linspace(np.nanmin(plotZ), np.nanmax(plotZ), 24)

    cf = ax.contourf(
        ALPHA_GRID,
        PHI_GRID * 100.0,
        plotZ,
        levels=levels_fill,
        cmap=cmap,
    )

    levels_line = np.linspace(np.nanmin(plotZ), np.nanmax(plotZ), 10)
    cs = ax.contour(
        ALPHA_GRID,
        PHI_GRID * 100.0,
        plotZ,
        levels=levels_line,
        colors="k",
        linewidths=0.6,
        alpha=0.45,
    )
    ax.clabel(cs, inline=True, fontsize=8, fmt="%.2f")

    ax.plot(
        result["best_alpha"],
        result["best_phi"] * 100.0,
        marker="*",
        markersize=13,
        color="red",
        markeredgecolor="black",
        linestyle="None",
        zorder=5,
    )

    # измеренная пористость образца
    ax.axhline(phi_sample * 100.0, linestyle="--", linewidth=1.2, color="black", alpha=0.6)

    ax.set_xscale("log")
    ax.set_title(title)
    ax.grid(True, alpha=0.18)

for ax in axes[2:]:
    ax.set_xlabel(r"Aspect ratio $\alpha$")

for ax in [axes[0], axes[2]]:
    ax.set_ylabel(r"Porosity, %")

cax = fig.add_axes([0.89, 0.16, 0.02, 0.68])
cbar = fig.colorbar(cf, cax=cax)
cbar.set_label(colorbar_label, rotation=90)

fig.suptitle(
    "OSP/GSA identifiability map in $(\\phi,\\alpha)$ space\n"
    f"Sample = {SAMPLE_ID}, measured porosity = {phi_sample*100:.2f}%",
    fontsize=17,
)

plt.show()

# fig.savefig("gsa_identifiability_phi_alpha.png", dpi=600, bbox_inches="tight")
# fig.savefig("gsa_identifiability_phi_alpha.pdf", bbox_inches="tight")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

# =========================================================
# Ожидается, что уже есть:
# - df
# - gsa_predict(matrix_tc, fluid_tc, phi, alpha)
# =========================================================

# -----------------------------
# Настройки
# -----------------------------
DEFAULT_AIR_TC = 0.025
DEFAULT_OIL_TC = 0.13
DEFAULT_BRINE_AVG_TC = 0.60

SAMPLE_ID_COL = "Sample"
POROSITY_COL = "Porosity,%"

MEASURED_COLS = {
    "dry": "TC air",
    "oil": "TC oil",
    "brine": "TC 60",
}

FLUID_TCS = {
    "dry": DEFAULT_AIR_TC,
    "oil": DEFAULT_OIL_TC,
    "brine": DEFAULT_BRINE_AVG_TC,
}

SAMPLE_ID = df[SAMPLE_ID_COL].iloc[0]

# Сетки параметров
ALPHA_GRID = np.logspace(-4, 0, 42)
LAMBDA_M_GRID = np.linspace(2.4, 3.3, 34)

PHI_RANGE_ABS = 0.06
PHI_MIN_GLOBAL = 0.01
PHI_MAX_GLOBAL = 0.30
N_PHI = 34

NORMALIZE_BY_OBS = True
PLOT_STATES = ["dry", "oil", "brine"]  # joint

# Оставляем только "хорошую" часть пространства
KEEP_PERCENTILE = 15   # нижние 15% по ΔJ

# -----------------------------
# Стиль
# -----------------------------
mpl.rcParams["font.family"] = "serif"
mpl.rcParams["font.serif"] = ["Times New Roman", "Times", "DejaVu Serif"]
mpl.rcParams["mathtext.fontset"] = "stix"

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
})

cmap = mpl.colormaps["YlGnBu_r"]

# -----------------------------
# Данные образца
# -----------------------------
sample_df = df.loc[df[SAMPLE_ID_COL] == SAMPLE_ID]
if sample_df.empty:
    raise ValueError(f"Sample {SAMPLE_ID!r} not found in df")

sample_row = sample_df.iloc[0]

phi_sample = float(sample_row[POROSITY_COL])
if phi_sample > 1.0:
    phi_sample = phi_sample / 100.0

obs = {
    "dry": float(sample_row[MEASURED_COLS["dry"]]),
    "oil": float(sample_row[MEASURED_COLS["oil"]]),
    "brine": float(sample_row[MEASURED_COLS["brine"]]),
}

phi_min = max(PHI_MIN_GLOBAL, phi_sample - PHI_RANGE_ABS)
phi_max = min(PHI_MAX_GLOBAL, phi_sample + PHI_RANGE_ABS)
PHI_GRID = np.linspace(phi_min, phi_max, N_PHI)

# -----------------------------
# Невязка
# -----------------------------
def single_state_misfit(pred, obs_value, normalize_by_obs=True):
    if normalize_by_obs:
        resid = (pred - obs_value) / obs_value
    else:
        resid = pred - obs_value
    return float(resid**2)

# -----------------------------
# Считаем 3D поле J(phi, alpha, lambda_m)
# -----------------------------
records = []

for phi in PHI_GRID:
    for lambda_m in LAMBDA_M_GRID:
        for alpha in ALPHA_GRID:
            total = 0.0
            for state in PLOT_STATES:
                pred = gsa_predict(
                    lambda_m,
                    FLUID_TCS[state],
                    float(phi),
                    float(alpha),
                )
                total += single_state_misfit(
                    pred,
                    obs[state],
                    normalize_by_obs=NORMALIZE_BY_OBS,
                )
            records.append((phi, alpha, lambda_m, total))

cloud_df = pd.DataFrame(records, columns=["phi", "alpha", "lambda_m", "J"])

Jmin = cloud_df["J"].min()
cloud_df["dJ"] = cloud_df["J"] - Jmin
cloud_df["plot_color"] = np.log10(1.0 + cloud_df["dJ"])

threshold = np.percentile(cloud_df["dJ"], KEEP_PERCENTILE)
plot_df = cloud_df.loc[cloud_df["dJ"] <= threshold].copy()

best_row = cloud_df.loc[cloud_df["J"].idxmin()]

# -----------------------------
# Построение 3D scatter
# -----------------------------
fig = plt.figure(figsize=(11, 8.5))
ax = fig.add_subplot(111, projection="3d")

sc = ax.scatter(
    plot_df["phi"] * 100.0,
    np.log10(plot_df["alpha"]),
    plot_df["lambda_m"],
    c=plot_df["plot_color"],
    cmap=cmap,
    s=18,
    alpha=0.75,
)

# лучшая точка
ax.scatter(
    [best_row["phi"] * 100.0],
    [np.log10(best_row["alpha"])],
    [best_row["lambda_m"]],
    color="red",
    edgecolor="black",
    s=120,
    marker="*",
)

# измеренная пористость образца
# тонкая направляющая линия
ax.plot(
    [phi_sample * 100.0, phi_sample * 100.0],
    [np.log10(ALPHA_GRID.min()), np.log10(ALPHA_GRID.min())],
    [LAMBDA_M_GRID.min(), LAMBDA_M_GRID.max()],
    linestyle="--",
    linewidth=1.0,
    color="black",
    alpha=0.4,
)

ax.set_xlabel("Porosity, %", labelpad=10)
ax.set_ylabel(r"$\log_{10}(\alpha)$", labelpad=10)
ax.set_zlabel(r"$\lambda_m$ (W m$^{-1}$ K$^{-1}$)", labelpad=10)

cbar = fig.colorbar(sc, pad=0.08, shrink=0.75)
cbar.set_label(r"$\log_{10}(1+\Delta J)$, where $\Delta J = J(\phi,\alpha,\lambda_m)-J_{\min}$")

ax.set_title(
    "OSP/GSA near-optimal parameter cloud\n"
    f"Sample = {SAMPLE_ID}, measured porosity = {phi_sample*100:.2f}%",
    pad=18,
)

ax.view_init(elev=190, azim=150)

plt.tight_layout()
plt.show()

# fig.savefig("gsa_identifiability_3d_phi_alpha_lambda_m.png", dpi=600, bbox_inches="tight")
# fig.savefig("gsa_identifiability_3d_phi_alpha_lambda_m.pdf", bbox_inches="tight")

In [ ]:
!pip -q install scikit-image opencv-python

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path

try:
    from skimage import io, color, filters, morphology, measure, exposure, util
    from skimage.segmentation import clear_border
    from skimage.color import rgb2hsv
    from scipy.ndimage import binary_fill_holes
except Exception as e:
    print("scikit-image (and friends) not available; skipping the image-processing section.")
    print("Install with: pip install scikit-image")
else:
    mpl.rcParams["font.family"] = "serif"
    mpl.rcParams["font.serif"] = ["Times New Roman", "Times", "DejaVu Serif"]
    mpl.rcParams["mathtext.fontset"] = "stix"

    plt.rcParams.update({
        "font.size": 12,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
    })

    # ---------------------------------------------------
    # Укажи свои 4 изображения
    # ---------------------------------------------------
    IMAGE_PATHS = [
        str(REPO_ROOT / "figures" / "Fig1a.png"),
        str(REPO_ROOT / "figures" / "Fig1b.png"),
        str(REPO_ROOT / "figures" / "Fig1c.png"),
        str(REPO_ROOT / "figures" / "Fig1d.png"),
    ]

    missing = [p for p in IMAGE_PATHS if not Path(p).exists()]
    if missing:
        print("Missing images (skipping this section):")
        for p in missing:
            print("  -", p)
    else:
        # ---------------------------------------------------
        # Если шлиф привязан к образцу, укажи sample_id и alpha_EMT
        # ---------------------------------------------------
        IMAGE_META = {
            "Fig1a.png": {"sample_id": "S1", "alpha_emt": 0.020},
            "Fig1b.png": {"sample_id": "S2", "alpha_emt": 0.008},
            "Fig1c.png": {"sample_id": "S3", "alpha_emt": 0.050},
            "Fig1d.png": {"sample_id": "S4", "alpha_emt": 0.015},
        }

        # ---------------------------------------------------
        # Настройки сегментации
        # mode = "dark"  -> поры темнее матрицы
        # mode = "blue"  -> поры окрашены в синий
        # ---------------------------------------------------
        SEGMENT_MODE = "blue"

        # Минимальный размер порового объекта в пикселях
        MIN_OBJ_AREA = 30

        # Порог crack-like объектов:
        # чем меньше aspect ratio, тем более трещиноподобная форма
        CRACK_AR_THRESHOLD = 0.15

        # Для выбора representative images:
        # 1) шлиф с median phi_2D
        # 2) шлиф с maximum crack fraction
        # 2) шлиф с maximum crack fraction

In [ ]:
def load_rgb(path):
    img = io.imread(path)
    if img.ndim == 2:
        img = np.stack([img, img, img], axis=-1)
    if img.shape[-1] == 4:
        img = img[..., :3]
    return util.img_as_float(img)


def segment_pores(img, mode="dark", min_obj_area_px=30):
    """
    Возвращает:
    pores_mask: bool mask пор/трещин
    matrix_mask: bool mask матрицы
    """
    if mode == "blue":
        hsv = rgb2hsv(img)
        h = hsv[..., 0]
        s = hsv[..., 1]
        v = hsv[..., 2]

        pores = (h > 0.50) & (h < 0.75) & (s > 0.20) & (v > 0.10)

    else:
        gray = color.rgb2gray(img)
        gray_eq = exposure.equalize_adapthist(gray, clip_limit=0.03)
        thr = filters.threshold_otsu(gray_eq)
        pores = gray_eq < thr

    pores = clear_border(pores)
    pores = morphology.remove_small_objects(pores, min_obj_area_px)
    pores = binary_fill_holes(pores)
    pores = morphology.binary_opening(pores, morphology.disk(1))
    pores = morphology.binary_closing(pores, morphology.disk(1))

    matrix = ~pores
    return pores.astype(bool), matrix.astype(bool)


def measure_pore_objects(pores_mask, um_per_pixel=1.0, crack_ar_threshold=0.15):
    label_img = measure.label(pores_mask)
    props = measure.regionprops_table(
        label_img,
        properties=[
            "label",
            "area",
            "major_axis_length",
            "minor_axis_length",
            "eccentricity",
            "orientation",
            "solidity",
            "centroid",
        ],
    )
    df = pd.DataFrame(props)

    if len(df) == 0:
        df["aspect_ratio"] = []
        df["is_crack"] = []
        df["orientation_deg"] = []
        df["area_um2"] = []
        df["major_axis_um"] = []
        df["minor_axis_um"] = []
        return df

    df = df[df["major_axis_length"] > 0].copy()

    # aspect ratio = b / a
    df["aspect_ratio"] = df["minor_axis_length"] / df["major_axis_length"]
    df["aspect_ratio"] = df["aspect_ratio"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["aspect_ratio"])

    # crack-like criterion
    df["is_crack"] = df["aspect_ratio"] < crack_ar_threshold

    # orientation в skimage в радианах; переведем в градусы [0,180)
    orient_deg = np.rad2deg(df["orientation"].to_numpy())
    orient_deg = (orient_deg + 180.0) % 180.0
    df["orientation_deg"] = orient_deg

    # перевод в физические единицы
    df["area_um2"] = df["area"] * (um_per_pixel ** 2)
    df["major_axis_um"] = df["major_axis_length"] * um_per_pixel
    df["minor_axis_um"] = df["minor_axis_length"] * um_per_pixel

    return df


def weighted_median(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    sorter = np.argsort(values)
    values = values[sorter]
    weights = weights[sorter]

    cum = np.cumsum(weights)
    cutoff = 0.5 * np.sum(weights)
    return values[np.searchsorted(cum, cutoff)]


def summarize_thin_section(image_name, pores_mask, obj_df, alpha_emt=None, um_per_pixel=1.0):
    pixel_area_um2 = um_per_pixel ** 2

    total_area = pores_mask.size
    pore_area = pores_mask.sum()

    phi_2d = pore_area / total_area if total_area > 0 else np.nan
    total_area_um2 = total_area * pixel_area_um2
    pore_area_um2 = pore_area * pixel_area_um2

    if len(obj_df) > 0:
        alpha_aw = weighted_median(obj_df["aspect_ratio"], obj_df["area_um2"])
        alpha_med = np.median(obj_df["aspect_ratio"])
        alpha_p25 = np.percentile(obj_df["aspect_ratio"], 25)
        alpha_p75 = np.percentile(obj_df["aspect_ratio"], 75)

        crack_area = obj_df.loc[obj_df["is_crack"], "area_um2"].sum()
        total_obj_area = obj_df["area_um2"].sum()
        crack_fraction = crack_area / total_obj_area if total_obj_area > 0 else np.nan

        dominant_orientation = weighted_median(obj_df["orientation_deg"], obj_df["area_um2"])

        mean_major_um = obj_df["major_axis_um"].mean()
        mean_minor_um = obj_df["minor_axis_um"].mean()
    else:
        alpha_aw = np.nan
        alpha_med = np.nan
        alpha_p25 = np.nan
        alpha_p75 = np.nan
        crack_fraction = np.nan
        dominant_orientation = np.nan
        mean_major_um = np.nan
        mean_minor_um = np.nan

    row = {
        "image": image_name,
        "phi_2D": phi_2d,
        "total_area_um2": total_area_um2,
        "pore_area_um2": pore_area_um2,
        "n_objects": len(obj_df),
        "alpha_img_aw": alpha_aw,
        "alpha_img_median": alpha_med,
        "alpha_img_p25": alpha_p25,
        "alpha_img_p75": alpha_p75,
        "crack_fraction": crack_fraction,
        "dominant_orientation_deg": dominant_orientation,
        "mean_major_um": mean_major_um,
        "mean_minor_um": mean_minor_um,
        "alpha_EMT": alpha_emt,
    }

    if alpha_emt is not None and np.isfinite(alpha_aw) and alpha_aw > 0:
        row["alpha_ratio_EMT_to_img"] = alpha_emt / alpha_aw
    else:
        row["alpha_ratio_EMT_to_img"] = np.nan

    return row

In [ ]:
results = []
object_tables = {}
masks = {}
images = {}

for path_str in IMAGE_PATHS:
    path = Path(path_str)
    img = load_rgb(path)
    pores_mask, matrix_mask = segment_pores(img, mode=SEGMENT_MODE)
    obj_df = measure_pore_objects(pores_mask)

    meta = IMAGE_META.get(path.name, {})
    alpha_emt = meta.get("alpha_emt", np.nan)

    summary_row = summarize_thin_section(
        image_name=path.name,
        pores_mask=pores_mask,
        obj_df=obj_df,
        alpha_emt=alpha_emt,
        um_per_pixel=UM_PER_PIXEL)

    if "sample_id" in meta:
        summary_row["sample_id"] = meta["sample_id"]

    results.append(summary_row)
    object_tables[path.name] = obj_df
    masks[path.name] = pores_mask
    images[path.name] = img

summary_df = pd.DataFrame(results)
display(summary_df)

In [ ]:
# 1) "типичный" шлиф: ближайший к median phi_2D
phi_med = summary_df["phi_2D"].median()
rep_typical_idx = (summary_df["phi_2D"] - phi_med).abs().idxmin()
rep_typical = summary_df.loc[rep_typical_idx, "image"]

# 2) "наиболее crack-rich" шлиф
rep_crack_idx = summary_df["crack_fraction"].idxmax()
rep_crack = summary_df.loc[rep_crack_idx, "image"]

representative_images = [rep_typical]
if rep_crack != rep_typical:
    representative_images.append(rep_crack)

print("Representative images:", representative_images)

In [ ]:
for image_name in images.keys():
    img = images[image_name]
    pores_mask = masks[image_name]
    obj_df = object_tables[image_name]

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))

    # 1. Original image
    axes[0].imshow(img)
    axes[0].set_title(f"{image_name}\nOriginal")
    axes[0].axis("off")

    # 2. Segmentation overlay
    overlay = img.copy()
    overlay[..., 0] = np.where(pores_mask, 1.0, overlay[..., 0])
    overlay[..., 1] = np.where(pores_mask, 0.2, overlay[..., 1])
    overlay[..., 2] = np.where(pores_mask, 0.2, overlay[..., 2])

    axes[1].imshow(overlay)
    axes[1].set_title("Pores / cracks segmentation")
    axes[1].axis("off")

    # 3. Aspect ratio distribution
    if len(obj_df) > 0:
        weight_col = "area_um2" if "area_um2" in obj_df.columns else "area"

        axes[2].hist(
            obj_df["aspect_ratio"],
            bins=25,
            weights=obj_df[weight_col],
            alpha=0.8
        )
        axes[2].axvline(
            np.median(obj_df["aspect_ratio"]),
            linestyle="--",
            label="median"
        )
        axes[2].axvline(
            weighted_median(obj_df["aspect_ratio"], obj_df[weight_col]),
            linestyle="-",
            label="area-weighted median"
        )
        axes[2].set_xlabel("Aspect ratio b/a")
        axes[2].set_ylabel("Area-weighted count")
        axes[2].set_title("Aspect ratio distribution")
        axes[2].legend(frameon=False)
    else:
        axes[2].text(0.5, 0.5, "No pore objects", ha="center", va="center")
        axes[2].set_axis_off()

    # 4. Orientation distribution
    if len(obj_df) > 0:
        weight_col = "area_um2" if "area_um2" in obj_df.columns else "area"
        bins = np.linspace(0, 180, 19)

        axes[3].hist(
            obj_df["orientation_deg"],
            bins=bins,
            weights=obj_df[weight_col],
            alpha=0.8
        )
        axes[3].set_xlabel("Orientation, deg")
        axes[3].set_ylabel("Area-weighted count")
        axes[3].set_title("Orientation distribution")
    else:
        axes[3].text(0.5, 0.5, "No pore objects", ha="center", va="center")
        axes[3].set_axis_off()

    fig.suptitle(image_name, fontsize=15)
    plt.tight_layout()
    plt.show()

In [ ]:
compare_df = summary_df.copy()

def qualitative_consistency(row):
    alpha_emt = row["alpha_EMT"]
    a25 = row["alpha_img_p25"]
    a75 = row["alpha_img_p75"]
    crack_fraction = row["crack_fraction"]

    if not np.isfinite(alpha_emt):
        return "no EMT alpha"
    if not (np.isfinite(a25) and np.isfinite(a75)):
        return "no image stats"

    if a25 <= alpha_emt <= a75:
        return "consistent"
    if alpha_emt < a25 and crack_fraction > 0.25:
        return "EMT more crack-like, but supported by crack-rich image"
    if alpha_emt < a25:
        return "EMT more crack-like than image median"
    if alpha_emt > a75:
        return "EMT more equant than image median"
    return "intermediate"

compare_df["consistency"] = compare_df.apply(qualitative_consistency, axis=1)

display(compare_df[[
    "image",
    "sample_id" if "sample_id" in compare_df.columns else "image",
    "phi_2D",
    "alpha_img_aw",
    "alpha_img_median",
    "alpha_img_p25",
    "alpha_img_p75",
    "crack_fraction",
    "dominant_orientation_deg",
    "alpha_EMT",
    "alpha_ratio_EMT_to_img",
    "consistency",
]])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import betaln
from scipy.optimize import minimize
from scipy.stats import beta

def fit_weighted_beta(x, w=None, eps=1e-6):
    """
    Weighted MLE fit for Beta(a, b) on x in (0, 1).
    """
    x = np.asarray(x, dtype=float)
    x = np.clip(x, eps, 1 - eps)

    if w is None:
        w = np.ones_like(x)
    else:
        w = np.asarray(w, dtype=float)

    # normalize weights
    w = w / np.sum(w)

    def neg_loglik(log_params):
        a = np.exp(log_params[0])
        b = np.exp(log_params[1])

        logpdf = (a - 1) * np.log(x) + (b - 1) * np.log(1 - x) - betaln(a, b)
        return -np.sum(w * logpdf)

    # стартовые значения
    x_mean = np.average(x, weights=w)
    x_var = np.average((x - x_mean)**2, weights=w)
    x_var = max(x_var, 1e-4)

    tmp = x_mean * (1 - x_mean) / x_var - 1
    a0 = max(x_mean * tmp, 1.0)
    b0 = max((1 - x_mean) * tmp, 1.0)

    res = minimize(
        neg_loglik,
        x0=np.log([a0, b0]),
        method="L-BFGS-B"
    )

    a_hat, b_hat = np.exp(res.x)
    return a_hat, b_hat, res


def plot_aspect_ratio_with_beta(obj_df, weight_col="area_um2", bins=25):
    x = obj_df["aspect_ratio"].to_numpy(dtype=float)
    w = obj_df[weight_col].to_numpy(dtype=float) if weight_col in obj_df.columns else np.ones_like(x)

    a_hat, b_hat, res = fit_weighted_beta(x, w=w)

    xx = np.linspace(1e-4, 0.9999, 400)
    pdf = beta.pdf(xx, a_hat, b_hat)

    # area-weighted histogram normalized as density
    plt.figure(figsize=(7, 4.8))
    plt.hist(
        x,
        bins=bins,
        weights=w,
        density=True,
        alpha=0.5,
        edgecolor="black",
        label="Area-weighted histogram"
    )
    plt.plot(xx, pdf, linewidth=2.5, label=fr"Beta fit: $\alpha={a_hat:.2f},\ \beta={b_hat:.2f}$")

    plt.xlabel("Aspect ratio $b/a$")
    plt.ylabel("Density")
    plt.title("Aspect ratio distribution with weighted Beta fit")
    plt.legend(frameon=False)
    plt.grid(True, alpha=0.25)
    plt.show()

    return a_hat, b_hat, res

In [ ]:
a_hat, b_hat, res = plot_aspect_ratio_with_beta(obj_df, weight_col="area_um2", bins=25)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.special import betaln
from scipy.optimize import minimize
from scipy.stats import beta
from pathlib import Path

mpl.rcParams["font.family"] = "serif"
mpl.rcParams["font.serif"] = ["Times New Roman", "Times", "DejaVu Serif"]
mpl.rcParams["mathtext.fontset"] = "stix"

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
})

# ---------------------------------------------------
# Weighted Beta fit on AO in (0,1)
# ---------------------------------------------------
def fit_weighted_beta(x, w=None, eps=1e-6):
    x = np.asarray(x, dtype=float)
    x = np.clip(x, eps, 1 - eps)

    if w is None:
        w = np.ones_like(x)
    else:
        w = np.asarray(w, dtype=float)

    mask = np.isfinite(x) & np.isfinite(w) & (w > 0)
    x = x[mask]
    w = w[mask]

    if len(x) == 0:
        return np.nan, np.nan, None

    w = w / np.sum(w)

    def neg_loglik(log_params):
        a = np.exp(log_params[0])
        b = np.exp(log_params[1])
        logpdf = (a - 1.0) * np.log(x) + (b - 1.0) * np.log(1.0 - x) - betaln(a, b)
        return -np.sum(w * logpdf)

    m = np.sum(w * x)
    v = np.sum(w * (x - m) ** 2)
    v = max(v, 1e-4)

    tmp = m * (1 - m) / v - 1
    a0 = max(m * tmp, 1.2)
    b0 = max((1 - m) * tmp, 1.2)

    res = minimize(
        neg_loglik,
        x0=np.log([a0, b0]),
        method="L-BFGS-B"
    )

    a_hat, b_hat = np.exp(res.x)
    return a_hat, b_hat, res

# ---------------------------------------------------
# Beta density in log10(AO) space
# if AO ~ Beta(a,b), z = log10(AO)
# p_z(z) = p_A(10^z) * 10^z * ln(10)
# ---------------------------------------------------
def beta_density_in_log10_space(z, a, b):
    A = 10 ** z
    pdf_A = beta.pdf(A, a, b)
    jac = A * np.log(10.0)
    return pdf_A * jac

# ---------------------------------------------------
# Weighted median
# ---------------------------------------------------
def weighted_median(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    sorter = np.argsort(values)
    values = values[sorter]
    weights = weights[sorter]

    cum = np.cumsum(weights)
    cutoff = 0.5 * np.sum(weights)
    return values[np.searchsorted(cum, cutoff)]

# ---------------------------------------------------
# Main plotting + parameter extraction
# ---------------------------------------------------
def plot_beta_logAO_individual_and_overall(
    object_tables,
    image_meta=None,
    image_names=None,
    weight_col_preferred="area_um2",
    bins=25,
    eps=1e-4,
):
    if image_names is None:
        image_names = list(object_tables.keys())

    # ---------------------------------------------
    # Collect global log-range
    # ---------------------------------------------
    all_log_ar = []
    for image_name in image_names:
        obj_df = object_tables[image_name]
        if len(obj_df) == 0 or "aspect_ratio" not in obj_df.columns:
            continue
        ar = np.asarray(obj_df["aspect_ratio"], dtype=float)
        ar = np.clip(ar, eps, 1.0)
        all_log_ar.append(np.log10(ar))

    if len(all_log_ar) == 0:
        raise ValueError("No aspect_ratio data found.")

    all_log_ar = np.concatenate(all_log_ar)
    xmin = np.floor(np.min(all_log_ar) * 2) / 2
    xmax = np.ceil(np.max(all_log_ar) * 2) / 2
    z_grid = np.linspace(xmin, xmax, 500)

    # ---------------------------------------------
    # Individual fits
    # ---------------------------------------------
    rows = []

    n = len(image_names)
    ncols = 2
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.8 * nrows), sharex=True, sharey=True)
    axes = np.atleast_1d(axes).ravel()

    pooled_ar = []
    pooled_w = []

    for ax, image_name in zip(axes, image_names):
        obj_df = object_tables[image_name]

        if len(obj_df) == 0 or "aspect_ratio" not in obj_df.columns:
            ax.text(0.5, 0.5, "No pore objects", ha="center", va="center")
            ax.set_axis_off()
            continue

        ar = np.asarray(obj_df["aspect_ratio"], dtype=float)
        ar = np.clip(ar, eps, 1.0)
        log_ar = np.log10(ar)

        if weight_col_preferred in obj_df.columns:
            weights = np.asarray(obj_df[weight_col_preferred], dtype=float)
        elif "area" in obj_df.columns:
            weights = np.asarray(obj_df["area"], dtype=float)
        else:
            weights = np.ones_like(ar)

        pooled_ar.append(ar)
        pooled_w.append(weights)

        # histogram
        ax.hist(
            log_ar,
            bins=bins,
            range=(xmin, xmax),
            weights=weights,
            density=True,
            alpha=0.45,
            edgecolor="black",
            linewidth=0.5,
            label="Area-weighted histogram",
        )

        # beta fit
        a_hat, b_hat, res = fit_weighted_beta(ar, w=weights)
        dens_z = beta_density_in_log10_space(z_grid, a_hat, b_hat)
        ax.plot(
            z_grid,
            dens_z,
            linewidth=2.2,
            label=fr"Beta fit: $a={a_hat:.2f},\, b={b_hat:.2f}$"
        )

        # summary lines
        med_log = np.median(log_ar)
        ax.axvline(med_log, linestyle="--", linewidth=1.2, label="Median")

        weight_col = weight_col_preferred if weight_col_preferred in obj_df.columns else "area"
        aw_med_linear = weighted_median(obj_df["aspect_ratio"], obj_df[weight_col])
        aw_med_log = np.log10(np.clip(aw_med_linear, eps, 1.0))
        ax.axvline(aw_med_log, linestyle="-", linewidth=1.5, label="Area-weighted median")

        alpha_emt = np.nan
        if image_meta is not None and image_name in image_meta:
            alpha_emt = image_meta[image_name].get("alpha_emt", np.nan)
            if np.isfinite(alpha_emt) and alpha_emt > 0:
                ax.axvline(
                    np.log10(alpha_emt),
                    linestyle=":",
                    linewidth=2.0,
                    label=r"$\log_{10}(\alpha_{EMT})$"
                )

        ax.set_title(image_name)
        ax.set_xlabel(r"$\log_{10}(AO)$, where $AO=b/a$")
        ax.set_ylabel("Density")
        ax.grid(True, alpha=0.25)

        rows.append({
            "image": image_name,
            "n_objects": len(obj_df),
            "beta_a": a_hat,
            "beta_b": b_hat,
            "alpha_img_weighted_median": aw_med_linear,
            "alpha_img_median": np.median(ar),
            "alpha_emt": alpha_emt,
        })

    for k in range(len(image_names), len(axes)):
        axes[k].set_visible(False)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.01))

    fig.suptitle("Individual log-aspect-ratio distributions with Beta fit", fontsize=16)
    fig.tight_layout(rect=[0, 0.06, 1, 0.96])
    plt.show()

    # ---------------------------------------------
    # Overall pooled fit
    # ---------------------------------------------
    pooled_ar = np.concatenate(pooled_ar)
    pooled_w = np.concatenate(pooled_w)

    pooled_log_ar = np.log10(np.clip(pooled_ar, eps, 1.0))
    a_all, b_all, res_all = fit_weighted_beta(pooled_ar, w=pooled_w)
    dens_all = beta_density_in_log10_space(z_grid, a_all, b_all)

    fig, ax = plt.subplots(1, 1, figsize=(8, 5.2))

    ax.hist(
        pooled_log_ar,
        bins=bins,
        range=(xmin, xmax),
        weights=pooled_w,
        density=True,
        alpha=0.45,
        edgecolor="black",
        linewidth=0.5,
        label="Pooled area-weighted histogram",
    )
    ax.plot(
        z_grid,
        dens_all,
        linewidth=2.5,
        label=fr"Pooled Beta fit: $a={a_all:.2f},\, b={b_all:.2f}$"
    )

    pooled_aw_med = weighted_median(pooled_ar, pooled_w)
    ax.axvline(
        np.log10(np.clip(pooled_aw_med, eps, 1.0)),
        linestyle="-",
        linewidth=1.8,
        label="Pooled area-weighted median"
    )
    ax.axvline(
        np.median(pooled_log_ar),
        linestyle="--",
        linewidth=1.4,
        label="Pooled median"
    )

    ax.set_title("Overall pooled log-aspect-ratio distribution")
    ax.set_xlabel(r"$\log_{10}(AO)$, where $AO=b/a$")
    ax.set_ylabel("Density")
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()

    rows.append({
        "image": "ALL_SAMPLES_POOLED",
        "n_objects": len(pooled_ar),
        "beta_a": a_all,
        "beta_b": b_all,
        "alpha_img_weighted_median": pooled_aw_med,
        "alpha_img_median": np.median(pooled_ar),
        "alpha_emt": np.nan,
    })

    params_df = pd.DataFrame(rows)
    return params_df

In [ ]:
image_names = [Path(p).name for p in IMAGE_PATHS]

beta_params_df = plot_beta_logAO_individual_and_overall(
    object_tables=object_tables,
    image_meta=IMAGE_META,
    image_names=image_names,
    weight_col_preferred="area_um2",
    bins=25,
)

display(beta_params_df)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import lil_matrix, csr_matrix
from scipy.sparse.linalg import spsolve
from skimage.transform import resize

# =========================================================
# ОЖИДАЕТСЯ, ЧТО УЖЕ ЕСТЬ:
# pores_mask : 2D bool array
# True  -> pores/cracks
# False -> matrix
# =========================================================

# -----------------------------
# Теплопроводности фаз
# -----------------------------
LAMBDA_M = 2.98
LAMBDA_AIR = 0.025
LAMBDA_OIL = 0.13
LAMBDA_BRINE = 0.60

# Если изображение очень большое, можно уменьшить
MAX_NX = 350   # удобно для Colab; можно увеличить

# =========================================================
# Подготовка mask
# =========================================================
def prepare_mask(mask, max_nx=350):
    mask = mask.astype(bool)

    ny, nx = mask.shape
    if max(nx, ny) <= max_nx:
        return mask

    scale = max_nx / max(nx, ny)
    new_ny = max(20, int(round(ny * scale)))
    new_nx = max(20, int(round(nx * scale)))

    resized = resize(
        mask.astype(float),
        (new_ny, new_nx),
        order=0,
        preserve_range=True,
        anti_aliasing=False,
    )
    return resized > 0.5

# =========================================================
# Поле теплопроводности
# =========================================================
def build_lambda_field(pores_mask, lambda_matrix, lambda_pore):
    lam = np.full(pores_mask.shape, lambda_matrix, dtype=float)
    lam[pores_mask] = lambda_pore
    return lam

# =========================================================
# Гармоническое среднее на грани
# =========================================================
def harmonic_mean(a, b, eps=1e-30):
    return 2.0 * a * b / (a + b + eps)

# =========================================================
# Решение ∇·(λ∇T)=0
#
# axis="x":
#   T=1 на левой границе
#   T=0 на правой границе
#   no-flux сверху/снизу
#
# axis="y":
#   T=1 сверху
#   T=0 снизу
#   no-flux слева/справа
# =========================================================
def solve_temperature_field(lam, axis="x"):
    ny, nx = lam.shape
    N = ny * nx

    def idx(i, j):
        return i * nx + j

    A = lil_matrix((N, N), dtype=float)
    b = np.zeros(N, dtype=float)

    for i in range(ny):
        for j in range(nx):
            p = idx(i, j)

            # -----------------------------------------
            # Dirichlet границы
            # -----------------------------------------
            if axis == "x":
                if j == 0:
                    A[p, p] = 1.0
                    b[p] = 1.0
                    continue
                elif j == nx - 1:
                    A[p, p] = 1.0
                    b[p] = 0.0
                    continue
            elif axis == "y":
                if i == 0:
                    A[p, p] = 1.0
                    b[p] = 1.0
                    continue
                elif i == ny - 1:
                    A[p, p] = 1.0
                    b[p] = 0.0
                    continue

            center = 0.0

            # -----------------------------------------
            # Соседи + no-flux на не-Dirichlet границах
            # -----------------------------------------

            # left
            if j > 0:
                kf = harmonic_mean(lam[i, j], lam[i, j-1])
                A[p, idx(i, j-1)] = -kf
                center += kf

            # right
            if j < nx - 1:
                kf = harmonic_mean(lam[i, j], lam[i, j+1])
                A[p, idx(i, j+1)] = -kf
                center += kf

            # up
            if i > 0:
                kf = harmonic_mean(lam[i, j], lam[i-1, j])
                A[p, idx(i-1, j)] = -kf
                center += kf

            # down
            if i < ny - 1:
                kf = harmonic_mean(lam[i, j], lam[i+1, j])
                A[p, idx(i+1, j)] = -kf
                center += kf

            A[p, p] = center

    A = csr_matrix(A)
    T = spsolve(A, b).reshape((ny, nx))
    return T

# =========================================================
# Эффективная теплопроводность
# =========================================================
def effective_conductivity_from_T(lam, T, axis="x"):
    ny, nx = lam.shape

    if axis == "x":
        # средний поток через левую внутреннюю грань
        q_vals = []
        jL = 0
        jR = 1
        for i in range(ny):
            kf = harmonic_mean(lam[i, jL], lam[i, jR])
            # dx = 1, Delta T imposed = 1 across length L = nx-1
            q = -kf * (T[i, jR] - T[i, jL])
            q_vals.append(q)

        q_mean = np.mean(q_vals)
        L = nx - 1
        deltaT = 1.0
        lambda_eff = q_mean * L / deltaT
        return float(lambda_eff)

    elif axis == "y":
        q_vals = []
        iT = 0
        iB = 1
        for j in range(nx):
            kf = harmonic_mean(lam[iT, j], lam[iB, j])
            q = -kf * (T[iB, j] - T[iT, j])
            q_vals.append(q)

        q_mean = np.mean(q_vals)
        L = ny - 1
        deltaT = 1.0
        lambda_eff = q_mean * L / deltaT
        return float(lambda_eff)

    else:
        raise ValueError("axis must be 'x' or 'y'")

# =========================================================
# Полный прогон для одного флюида
# =========================================================
def simulate_2d_conductivity(pores_mask, lambda_matrix, lambda_pore):
    mask_small = prepare_mask(pores_mask, max_nx=MAX_NX)
    lam = build_lambda_field(mask_small, lambda_matrix, lambda_pore)

    Tx = solve_temperature_field(lam, axis="x")
    Ty = solve_temperature_field(lam, axis="y")

    lam_eff_x = effective_conductivity_from_T(lam, Tx, axis="x")
    lam_eff_y = effective_conductivity_from_T(lam, Ty, axis="y")

    return {
        "mask": mask_small,
        "lambda_field": lam,
        "Tx": Tx,
        "Ty": Ty,
        "lambda_eff_x": lam_eff_x,
        "lambda_eff_y": lam_eff_y,
    }

# =========================================================
# Запуск: dry / oil / brine
# =========================================================
results_2d = {
    "dry": simulate_2d_conductivity(pores_mask, LAMBDA_M, LAMBDA_AIR),
    "oil": simulate_2d_conductivity(pores_mask, LAMBDA_M, LAMBDA_OIL),
    "brine": simulate_2d_conductivity(pores_mask, LAMBDA_M, LAMBDA_BRINE),
}

summary_2d = []
for state, res in results_2d.items():
    summary_2d.append({
        "state": state,
        "lambda_eff_x_2D": res["lambda_eff_x"],
        "lambda_eff_y_2D": res["lambda_eff_y"],
        "anisotropy_ratio_x_over_y": res["lambda_eff_x"] / res["lambda_eff_y"] if res["lambda_eff_y"] != 0 else np.nan,
    })

summary_2d = np.array(summary_2d, dtype=object)

for row in summary_2d:
    print(row)

In [ ]:
import pandas as pd

summary_df_2d = pd.DataFrame(list(summary_2d))
display(summary_df_2d)

In [ ]:
state_to_show = "dry"   # "dry", "oil", "brine"
res = results_2d[state_to_show]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].imshow(res["mask"], cmap="gray")
axes[0].set_title(f"{state_to_show}: pore geometry")
axes[0].axis("off")

im1 = axes[1].imshow(res["Tx"], cmap="inferno")
axes[1].set_title(r"$T(x)$ loading")
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(res["Ty"], cmap="inferno")
axes[2].set_title(r"$T(y)$ loading")
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.show()